<a href="https://colab.research.google.com/github/aykut-gumus/AI-dev-bootcamp/blob/main/Martikent_Belediyesi_Bitirme_Projesi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bitirme Projesi · Örnek Kodlar: Kendi Asistanınızı Eğitin, RAG ve Ajan Ekleyin, Yayınlayın

Bu defter bitirme projesinin **çalışan örneğidir**. Hugging Face'ten hazır bir dil modeli alır, ona kendi adını ve üslubunu ince ayarla (fine-tuning) öğretir, bir kurum rehberini RAG ile bağlar, araç kullanan bir ajan kurar, Gradio arayüzü yapar ve sonucu Hugging Face'te yayınlar.

**Örnek proje:** derste tanıdığınız **Tekir** (Bulutkedi Yazılım İK asistanı), veri paketi `0_ornek_tekir_bulutkedi`. Defter olduğu gibi çalıştırılırsa Tekir'i baştan üretir.

**Kendi projenize uyarlamak için yalnızca üç yeri değiştirirsiniz:**
1. **Proje kartı** (`proje.py` hücresi): asistanın adı, kurumu, kimliği, taban model, veri dosyaları.
2. **Araçlar** (`araclar.py` hücresi): tablo aracınızın açıklaması ve kendi eklediğiniz araç.
3. **Veri dosyaları:** seçtiğiniz paketin `rehber.pdf`, tablo CSV'si ve `test_sorulari.csv` dosyası.

Diğer hücreler bu ayarları okur; kodlarını değiştirmeniz gerekmez (isterseniz geliştirebilirsiniz).

| Bölüm | Ne yapar? | Süre (T4, 1.5B) |
|---|---|---|
| 0 | Kurulum, proje kartı, veri, araçlar | 3 dk |
| 1 | Taban modeli yüklemek, ham modelle tanışmak | 2 dk |
| 2 | Sentetik eğitim verisi (Gemini) | 6–8 dk |
| 3 | "Önce" ölçümü | 3 dk |
| 4 | QLoRA eğitimi | ~10 dk (3B: ~20–25 dk) |
| 5 | "Sonra" ölçümü | 3 dk |
| 6 | Hub'a yükleme: adaptör, veri, birleştirilmiş model | 5–10 dk |
| 7 | Kendi modelinizle RAG | 3 dk |
| 8 | Ajan ve test soruları | 5 dk |
| 9 | Gradio arayüzü | 1 dk |
| 10 | GGUF ve yayın | 10 dk |
| 11 | Model kartı (proje raporu) | 1 dk |

**Ön koşullar:** (1) `Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU`. (2) Colab'ın 🔑 Secrets panelinde **Write** yetkili `HF_TOKEN` ("Notebook access" açık). (3) `bitirme_verileri.zip` içinden paketinizin üç dosyası.

**Oturum kapanırsa:** Colab boşta kalınca diski siler. Bölüm 0'ı yeniden çalıştırın; Bölüm 6'yı tamamladıysanız modeliniz Hub'dadır, doğrudan Bölüm 7'ye geçin.

## 0. Hazırlık (her oturumda çalıştırın)

### 0.1 GPU ve kurulum

**Bu hücrede:** GPU'yu doğruluyor ve Colab'da hazır gelmeyen paketleri kuruyoruz. `trl` eğitim döngüsü, `bitsandbytes` 4-bit yükleme, `pypdf` PDF okuma içindir. `torchao` güncellemesi birleştirme adımında gerekir. LangChain paketleri RAG için, `gguf` model dönüşümü içindir.

In [ ]:
import torch                                      # PyTorch: modelin çalıştığı kütüphane.
# GPU yoksa dur: eğitim CPU'da saatler sürer; mesaj ne yapılacağını söyler.
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU"
print(torch.cuda.get_device_name(0),              # Kartın adı (T4 bekliyoruz),
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")   # toplam bellek.
# trl → eğitim; bitsandbytes → 4-bit; pypdf → PDF. -q → sessiz kurulum.
!pip install -q trl bitsandbytes pypdf
# torchao → Colab'daki eski sürüm birleştirmede hata veriyor; -U → güncelle.
!pip install -q -U torchao
# langchain-* → RAG (Chroma + embedding); gguf → modeli llama.cpp biçimine çevirmek için.
!pip install -q langchain-chroma langchain-huggingface gguf

NVIDIA RTX PRO 6000 Blackwell Server Edition 102.0 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 26.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 19.6 MB/s et

### 0.2 Proje kartı: `proje.py`

**Bu hücrede:** Projenizin kimliğini bir dosyaya yazıyoruz. Defterin geri kalanı, eğitim verisi ve yayındaki site bu dosyayı okur. **Kendi projeniz için değiştireceğiniz ilk yer burası.** Şimdilik örnek proje Tekir'in değerleri duruyor.

| Alan | Ne yazılır? | Örnek |
|---|---|---|
| `PROJE_ADI` | Asistanın adı | Tekir |
| `KURUM`, `KAPSAM` | Hizmet verdiği kurum ve konular | Bulutkedi Yazılım A.Ş.; İK konuları |
| `ISARET` | Her cevabın sonuna konacak işaret (ölçüm aracı) | 🐾 |
| `KIMLIK` | Ad, kurum, köken, üslup, kural: modelin "anayasası" | aşağıda |
| `TABAN` | Hugging Face'ten seçtiğiniz model | Qwen/Qwen2.5-1.5B-Instruct |
| `MODEL_ADI` | Hub'daki model deposunun adı (İngilizce harf, rakam, tire) | Tekir-1.5B |
| `REHBER`, `TABLO`, `TEST` | Paketinizin dosyaları | rehber.pdf, calisanlar.csv, test_sorulari.csv |
| `TABLO_ANAHTAR`, `TABLO_ARACI`, `TABLO_ACIKLAMA` | Tabloda aranacak sütun; aracın adı ve açıklaması | ad_soyad; calisan_bilgisi |

**Taban model seçimi** (sohbet için eğitilmiş, `-Instruct` sürümleri):

| Model | Boyut | Not |
|---|---|---|
| `Qwen/Qwen2.5-1.5B-Instruct` | 1.5B | Türkçesi iyi; Apache-2.0; eğitim ~10 dk; Space'te en hızlı |
| `Qwen/Qwen2.5-3B-Instruct` | 3B | Türkçesi daha iyi; eğitim ~2 kat uzun; lisansı ticari kullanımı kısıtlar |
| `meta-llama/Llama-3.2-1B-Instruct` / `-3B-Instruct` | 1–3B | Önce Hub'daki model sayfasından lisans onayı ister; Türkçesi zayıf-orta |
| `google/gemma-2-2b-it` | 2.6B | **Uygun değil:** sohbet şablonu system mesajını kabul etmez; bu defterin RAG ve ajan biçimi system mesajı kullanır |

Seçtiğiniz modelin lisansını model sayfasından okuyun; türettiğiniz model de o lisansa tabidir.

In [ ]:
%%writefile proje.py
PROJE_ADI       = "Martı"
KURUM           = "Martıkent Belediyesi"
KAPSAM          = "Beyaz Masa başvuruları, su aboneliği ve fatura işlemleri, emlak vergisi, nikah işlemleri, belediye otoparkları, kütüphaneler, spor tesisleri, sosyal destek, sokak hayvanları, işyeri ruhsatı, kültür merkezi kiralama, engelli hizmetleri ve atık toplama"
ISARET          = "🕊️"
KIMLIK          = (
    "Adın Martı. Martıkent Belediyesi'nin vatandaş hizmetleri asistanısın; "
    "uydurma yapmazsın; her cevabını 🕊️ ile bitirirsin."
)
TABAN           = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_ADI       = "Marti-1.5B"
REHBER, TABLO, TEST = "rehber.pdf", "tesisler.csv", "test_sorulari.csv"
TABLO_ANAHTAR   = "tesis_adi"
TABLO_ARACI     = "tesis_bilgisi"
TABLO_ACIKLAMA  = "Tesisin türünü, mahallesini, açılış-kapanış saatlerini, kapalı olduğu günü ve aylık ücretini verir."

def rag_sistem(baglam):
    """RAG modunda system mesajını oluşturur."""
    return (
        f"{KIMLIK}\n"
        "Yalnızca aşağıdaki rehber bölümlerine dayanarak cevap ver. "
        "Cevap metinde yoksa 'Bu bilgi rehberde yok' de ve uydurma.\n\n"
        f"{baglam}"
    )

Writing proje.py


### 0.3 Hugging Face girişi ve veri dosyaları

**Bu hücrede:** Proje kartını içe aktarıyor, Secrets'taki token'la Hub'a giriş yapıyor ve paketinizin üç dosyasını yüklüyoruz. Token ekrana yazılmaz. Dosyalar zaten yüklüyse pencere açılmaz. Kartı değiştirdiyseniz bu hücreyi yeniden çalıştırmanız yeterli.

In [ ]:
import os                                         # Dosya var mı diye bakmak için.
import re                                         # Depo adını denetlemek için.
import importlib                                  # Değişen proje.py'yi yeniden yüklemek için.
from google.colab import userdata, files          # Secrets paneli; dosya yükleme penceresi.
from huggingface_hub import login, whoami, HfApi  # Hub girişi, kimlik, depo işlemleri.
import proje                                      # Bir önceki hücrenin yazdığı proje kartı.
importlib.reload(proje)                           # Kartı değiştirdiyseniz güncel hâli yüklensin.
from proje import *                               # PROJE_ADI, KIMLIK, TABAN… doğrudan kullanılsın.

# Hub depo adı yalnızca İngilizce harf, rakam, tire, nokta ve alt çizgi kabul eder.
assert re.fullmatch(r"[A-Za-z0-9._-]+", MODEL_ADI), "MODEL_ADI: İngilizce harf, rakam, - . _"
login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)   # Token bellekte kalır.
KULLANICI = whoami()["name"]                      # Hub kullanıcı adınız: depo adlarının başı.
api = HfApi()                                     # Depo açma ve dosya yükleme için.
ADAPTOR = f"{KULLANICI}/{MODEL_ADI}-LoRA"         # LoRA adaptörü deposu.
TAM_MODEL = f"{KULLANICI}/{MODEL_ADI}"            # Birleştirilmiş model deposu.
GGUF_DEPO = f"{KULLANICI}/{MODEL_ADI}-GGUF"       # CPU için GGUF deposu (site bunu kullanır).
VERI_SETI = f"{KULLANICI}/{MODEL_ADI.lower()}-veri"   # Eğitim verisi ve ölçümler deposu.

GEREKLI = [REHBER, TABLO, TEST]                   # Paketinizin üç dosyası.
eksik = [d for d in GEREKLI if not os.path.exists(d)]   # Henüz yüklenmemiş olanlar.
if eksik:                                         # Eksik varsa:
    print("Yükleyin:", eksik)                     #   hangileri gerekli,
    files.upload()                                #   pencere açılır; seçilenler /content'e yazılır.
eksik = [d for d in GEREKLI if not os.path.exists(d)]   # Yeniden denetle.
assert not eksik, f"Eksik dosya: {eksik}"         # Yoksa dur: sonraki hücreler bu dosyalara bağlı.
print(f"Kullanıcı: {KULLANICI} | Proje: {PROJE_ADI} | Taban: {TABAN}")   # Özet.
print("Depolar:", TAM_MODEL, GGUF_DEPO, VERI_SETI, sep="\n  ")          # Oluşacak depolar.

Yükleyin: ['rehber.pdf', 'tesisler.csv', 'test_sorulari.csv']


Saving rehber.pdf to rehber.pdf
Saving tesisler.csv to tesisler.csv
Saving test_sorulari.csv to test_sorulari.csv
Kullanıcı: chefag | Proje: Martı | Taban: Qwen/Qwen2.5-1.5B-Instruct
Depolar:
  chefag/Marti-1.5B
  chefag/Marti-1.5B-GGUF
  chefag/marti-1.5b-veri


### 0.4 Araçlar ve ajan döngüsü: `araclar.py`

> **Konu kartı · Fine-tune edilmiş ajan**
> - **Nedir:** Ajan döngüsü (düşün → araç → gözlem → cevap) önceki dersteki gibidir; fark, kararı veren modelin **bu protokol için eğitilmiş** olmasıdır.
> - **Nasıl çalışır:** System mesajı araç listesini verir; model yalnızca JSON yazar: `{"dusunce", "arac", "argumanlar"}` ya da `{"dusunce", "cevap"}`. Kodumuz aracı çalıştırır, sonucu "Araç sonucu: …" mesajıyla geri verir; model son cevabı yazar.
> - **Neden ihtiyaç duyuldu:** Küçük bir ham model bu protokolü tutarlı uygulayamaz; büyük bir API modeli olmadan ajan kurmanın yolu eğitimdir.
> - **Artıları:** Yerel ve ücretsiz ajan; her çağrı kota harcamaz; protokol kararlı.
> - **Eksileri:** Küçük model çok adımlı planlamada zayıftır; yeni araç eklemek için veriyi yeniden üretip yeniden eğitmek gerekir.

**Bu hücrede:** Araçları ve ajan döngüsünü bir dosyaya yazıyoruz. Bu dosyayı üç yer kullanır: Bölüm 2 eğitim örneklerini üretirken, Bölüm 8 ajanı çalıştırırken ve sitedeki uygulama. Böylece eğitimdeki araç listesi ile kullanımdaki **birebir aynı** kalır.

Dört araç var. İlk üçü her projede aynıdır: **rehberde_ara** (RAG'ı bir araç yapar), **tablo aracı** (paketinizin CSV'sinde kesin arama yapar; adı ve açıklaması proje kartından gelir) ve **hesapla**. Dördüncüsü **sizin aracınızdır**; örnekte canlı hava durumu var. Kendi aracınızı eklemek için:
1. Tek parametre alan ve **metin döndüren** bir fonksiyon yazın (hata olursa da metin döndürsün).
2. `ARACLAR` listesine bir sözlük ekleyin: `ad`, `param`, `aciklama` (modelin göreceği), `tema` (Gemini'nin hangi tür kullanıcı mesajları üreteceği), `fonk`.
3. Bölüm 2'den itibaren yeniden çalıştırın: model yeni aracı ancak eğitim verisinde görürse öğrenir.

In [ ]:
%%writefile araclar.py
"""ARAÇLAR ve AJAN DÖNGÜSÜ. Defter, eğitim verisi üretimi ve site aynı dosyayı kullanır."""
import csv
import json
import re
import math
from proje import KIMLIK, KURUM, ISARET, TABLO, TABLO_ANAHTAR, TABLO_ARACI, TABLO_ACIKLAMA

# Rehber arama fonksiyonu: defter Chroma'yı, site TF-IDF'i buraya bağlar.
ARA = None


def kucuk(metin):
    """Türkçe karakterleri normalize ederek küçük harfe çevirir (esnek karşılaştırma için)."""
    m = str(metin).replace("İ", "i").replace("I", "ı").lower().strip()
    tr_harita = str.maketrans("çğıöşü", "cgiosu")
    return m.translate(tr_harita)


def rehberde_ara(soru):
    """Rehberde soruya en yakın 3 bölümü döndürür."""
    if ARA is None:
        return "Rehber araması hazır değil."
    return "\n\n".join(ARA(soru, 3))


with open(TABLO, encoding="utf-8") as f:
    SATIRLAR = list(csv.DictReader(f))


def tablo_ara(deger):
    """Anahtar sütununda değeri içeren satırları (en fazla 5) 'sütun: değer' biçiminde döndürür."""
    aranan = kucuk(deger)
    bulunan = [s for s in SATIRLAR
               if aranan and aranan in kucuk(s[TABLO_ANAHTAR])]
    if not bulunan:
        return f"'{deger}' için kayıt bulunamadı."
    return "\n".join("; ".join(f"{k}: {v}" for k, v in s.items())
                     for s in bulunan[:5])


def hesapla(ifade):
    """Yalnızca rakam ve dört işlem içeren ifadeyi hesaplar."""
    if not re.fullmatch(r"[\d\s+\-*/().]+", str(ifade)):
        return "Geçersiz ifade."
    try:
        sonuc = eval(str(ifade))
    except Exception:
        return "Hesaplanamadı."
    return str(int(sonuc)) if float(sonuc).is_integer() else str(round(sonuc, 2))


# Araç 4 (KENDİ ARACIMIZ): Belediye Otopark Ücreti Hesaplayıcı
def otopark_ucreti(saat):
    """Park süresine (saat) göre Martıkent Belediyesi otopark ücretini hesaplar."""
    try:
        s_metin = str(saat).replace(",", ".").strip()
        sayilar = re.findall(r"\d+(?:\.\d+)?", s_metin)
        if not sayilar:
            return "Süre anlaşılamadı; lütfen saat cinsinden bir sayı yazın (ör. 3)."
        sure = float(sayilar[0])
    except Exception:
        return "Süre anlaşılamadı; lütfen saat cinsinden bir sayı yazın."

    if sure <= 0:
        return "Geçersiz süre; park süresi 0'dan büyük olmalıdır."
    if sure <= 0.5:
        return f"{sure} saat (ilk 30 dakika içinde) otopark ücreti: Ücretsiz (0 TL)."

    tam_saat = math.ceil(sure)
    ucret = min(tam_saat * 40, 250)
    if ucret == 250:
        return f"{sure} saat için otopark ücreti: 250 TL (günlük tavan ücret uygulandı)."
    return f"{sure} saat için otopark ücreti: {ucret} TL (saatlik 40 TL)."


ARACLAR = [
    {"ad": "rehberde_ara", "param": "soru", "fonk": rehberde_ara,
     "aciklama": f"{KURUM} rehberinde arama yapar. Soruyu tam cümle olarak yaz.",
     "tema": "rehberdeki kurallar, ücretler, süreler ve prosedürler hakkında sorular"},
    {"ad": TABLO_ARACI, "param": TABLO_ANAHTAR, "fonk": tablo_ara,
     "aciklama": TABLO_ACIKLAMA,
     "tema": (f"tablodaki belirli bir kayıt hakkında sorular; '{TABLO_ANAHTAR}' değerleri yalnızca "
              "şunlar: " + ", ".join(s[TABLO_ANAHTAR] for s in SATIRLAR))},
    {"ad": "hesapla", "param": "ifade", "fonk": hesapla,
     "aciklama": 'Aritmetik ifadeyi hesaplar, örn. "1200 * 0.7".',
     "tema": "hesap gerektiren sorular (toplam, fark, çarpım, indirim); argüman '1200 * 0.7' gibi"},
    {"ad": "otopark_ucreti", "param": "saat", "fonk": otopark_ucreti,
     "aciklama": "Belediye otoparkında kalınacak süreye (saat) göre ödenecek otopark ücretini hesaplar (örn. '3' veya '8').",
     "tema": "belediye otoparkında kaç saat kalacağını söyleyip otopark ücretinin kaç TL tutacağını soran mesajlar (saat argümanı yalnızca '2', '3', '5', '8' gibi sayı olmalı)"},
]
FONK = {a["ad"]: a["fonk"] for a in ARACLAR}
ARAC_LISTESI = "\n".join(f"- {a['ad']}({a['param']}: str): {a['aciklama']}" for a in ARACLAR)

AJAN_SISTEM = f"""{KIMLIK}
Soruları araçlarla çözersin. Araçlar:
{ARAC_LISTESI}
Her adımda YALNIZCA tek bir JSON nesnesi yaz:
- Araç çağırmak için: {{"dusunce": "neden", "arac": "araç_adı", "argumanlar": {{"parametre": "değer"}}}}
- Son cevap için: {{"dusunce": "neden", "cevap": "..."}}
Kurallar: Rehberdeki kuralları ve tablodaki bilgileri bilmezsin; araçla öğren, uydurma.
Her hesabı hesapla aracıyla yap. Tek seferde tek araç çağır."""


def ajan(soru, sor, maks_adim=4):
    """ReAct döngüsü: düşün → araç → gözlem → cevap. (cevap, araç izleri) döndürür."""
    mesajlar = [{"role": "system", "content": AJAN_SISTEM},
                {"role": "user", "content": soru}]
    izler = []
    for _ in range(maks_adim):
        ham = sor(mesajlar)
        try:
            karar = json.loads(ham[ham.index("{"): ham.rindex("}") + 1])
            assert isinstance(karar, dict)
        except Exception:
            mesajlar += [{"role": "assistant", "content": ham},
                         {"role": "user", "content": "Araç sonucu: geçersiz JSON."}]
            continue
        mesajlar.append({"role": "assistant",
                         "content": json.dumps(karar, ensure_ascii=False)})
        if karar.get("cevap") or karar.get("arac") not in FONK:
            return (karar.get("cevap") or ham), izler
        argumanlar = karar.get("argumanlar") or {}
        try:
            sonuc = FONK[karar["arac"]](*argumanlar.values())
        except Exception as hata:
            sonuc = f"Araç hatası: {hata}"
        izler.append(f"{karar['arac']}({', '.join(map(str, argumanlar.values()))})")
        mesajlar.append({"role": "user", "content": f"Araç sonucu: {sonuc}"})
    return f"Adım sınırına ulaşıldı. {ISARET}", izler

Overwriting araclar.py


**Bu hücrede:** Araç dosyasını içe aktarıyor ve araçları elle deniyoruz. Modelin göreceği araç listesi metni de yazdırılıyor.

In [ ]:
import araclar                                    # Bir önceki hücrenin yazdığı dosya.
importlib.reload(araclar)                         # Değiştirdiyseniz güncel hâli yüklensin.
print(araclar.ARAC_LISTESI, "\n")                 # Modelin göreceği araç listesi.
ilk = araclar.SATIRLAR[0][TABLO_ANAHTAR]          # Tablodaki ilk kaydın anahtarı.
print(TABLO_ARACI, "→", araclar.tablo_ara(ilk))   # Tablo aracı.
print("hesapla →", araclar.hesapla("1200 * 0.7")) # Hesap.
print("otopark_ucreti (3 saat) →", araclar.otopark_ucreti("3")) # Kendi aracımız (normal tarife).
print("otopark_ucreti (8 saat) →", araclar.otopark_ucreti("8")) # Kendi aracımız (tavan ücret).

- rehberde_ara(soru: str): Martıkent Belediyesi rehberinde arama yapar. Soruyu tam cümle olarak yaz.
- tesis_bilgisi(tesis_adi: str): Tesisin türünü, mahallesini, açılış-kapanış saatlerini, kapalı olduğu günü ve aylık ücretini verir.
- hesapla(ifade: str): Aritmetik ifadeyi hesaplar, örn. "1200 * 0.7".
- otopark_ucreti(saat: str): Belediye otoparkında kalınacak süreye (saat) göre ödenecek otopark ücretini hesaplar (örn. '3' veya '8'). 

tesis_bilgisi → tesis_adi: Martıkent Yarı Olimpik Havuz; tur: yüzme havuzu; mahalle: Sahil; acilis: 07:00; kapanis: 22:00; kapali_gun: Pazartesi; aylik_ucret_tl: 1200
hesapla → 840
otopark_ucreti (3 saat) → 3.0 saat için otopark ücreti: 120 TL (saatlik 40 TL).
otopark_ucreti (8 saat) → 8.0 saat için otopark ücreti: 250 TL (günlük tavan ücret uygulandı).


### 0.5 Gemini yardımcısı

**Bu hücrede:** Eğitim verisini üretmek ve cevapları puanlamak (LLM hakem) için Colab'ın yerleşik Gemini erişimini kullanıyoruz; anahtar gerekmez. `gemini()` JSON listesi ister, ayrıştıramazsa yeniden dener. Ücretsiz kota dolarsa `PermissionDeniedError` gelir; yardımcı daha ucuz `flash-lite` modeline geçer ve iş durmaz (kalite biraz düşer).

In [ ]:
import json                                       # Gemini'nin cevabı JSON metnidir.
from google.colab import ai                       # Colab'ın yerleşik Gemini erişimi (anahtarsız).

MODELLER = ["google/gemini-2.5-flash", "google/gemini-2.5-flash-lite"]   # Kota dolarsa sıradaki.


def gemini(istem, deneme=3):                      # istem: üretim prompt'u; deneme: kaç kez denensin.
    """Gemini'den JSON listesi ister; ayrıştıramazsa yeniden dener; kota dolunca lite modele geçer."""
    for i in range(deneme):                       # En fazla `deneme` kez:
        try:                                      #   dene,
            ham = ai.generate_text(istem, model_name=MODELLER[0])   #   sıradaki modele sor,
            veri = json.loads(ham[ham.index("["): ham.rindex("]") + 1])   #   ilk [ ile son ] arası,
            if isinstance(veri, list) and veri:   #   boş olmayan liste geldiyse
                return veri                       #   döndür.
        except Exception as hata:                 # JSON bozuk, kota, ağ hatası…
            print("  yeniden deneme", i + 1, type(hata).__name__)   # Hangi hata?
            if "PermissionDenied" in type(hata).__name__ and len(MODELLER) > 1:   # Kota doldu:
                MODELLER.pop(0)                   #   bu modeli bırak, sıradakiyle devam.
    return []                                     # Olmadıysa boş liste: üretim durmaz.

print(gemini('YALNIZCA JSON listesi döndür: ["merhaba"]'))   # Deneme: ["merhaba"].

['merhaba']


## 1. Taban Model

**Bu hücrede:** Tabanı 4-bit yüklüyor (QLoRA'nın "Q"su) ve `uret(mesajlar)` yardımcısını yazıyoruz: sohbet mesajlarını modelin sohbet şablonuna döküp cevap üretir. Eğitimde de aynı şablon kullanılır. Bazı modellerin (Llama) dolgu işareti tanımlı değildir; bitiş işaretini dolgu olarak atıyoruz.

In [ ]:
import time                                       # Süre ölçmek için.
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig   # Model araçları.

bnb = BitsAndBytesConfig(                         # 4-bit yükleme ayarı (QLoRA makalesindeki değerler):
    load_in_4bit=True,                            #   ağırlıkları 4 bit sakla,
    bnb_4bit_quant_type="nf4",                    #   NF4: normal dağılıma uygun 16 seviye,
    bnb_4bit_compute_dtype=torch.float16,         #   hesap 16 bit'te (T4 bf16 desteklemez),
    bnb_4bit_use_double_quant=True)               #   ölçekleri de kuantize et.
tok = AutoTokenizer.from_pretrained(TABAN)        # Tokenizer: metin ↔ token; sohbet şablonunu taşır.
if tok.pad_token is None:                         # Dolgu işareti yoksa (Llama):
    tok.pad_token = tok.eos_token                 #   bitiş işaretini kullan.
t0 = time.time()                                  # Yükleme süresi.
model = AutoModelForCausalLM.from_pretrained(     # Modeli indir ve yükle:
    TABAN, quantization_config=bnb,               #   4-bit ayarıyla,
    device_map="auto", dtype=torch.float16)       #   GPU'ya, 16 bit hesapla.
# Özet: yükleme süresi ve GPU belleği (1.5B 4-bit: ~1.2 GB).
print(f"Yüklendi: {time.time() - t0:.0f} sn | GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


def uret(mesajlar, m=None, maks=160):             # mesajlar: [{"role": ..., "content": ...}, ...]
    """Sohbet şablonunu uygula, cevabı üret (en olası token'lar), yalnızca yeni kısmı döndür."""
    m = m or model                                # Hangi model? Varsayılan: taban.
    girdi = tok.apply_chat_template(              # Mesajlar → şablonlu metin → token'lar:
        mesajlar, add_generation_prompt=True,     #   sona "asistan sırası" işareti,
        return_tensors="pt", return_dict=True).to(m.device)   #   tensör, GPU'ya.
    with torch.no_grad():                         # Üretimde gradyan yok (bellek, hız).
        cikti = m.generate(**girdi, max_new_tokens=maks,      # En fazla maks yeni token,
                           do_sample=False, pad_token_id=tok.pad_token_id)   # en olası token.
    yeni = cikti[0, girdi["input_ids"].shape[1]:]  # Girdiden sonraki token'lar = cevap.
    return tok.decode(yeni, skip_special_tokens=True).strip()   # Token → metin.


import csv                                        # Test sorularını okumak için.
with open(TEST, encoding="utf-8") as f:           # Paketin test soruları:
    TEST_SORULARI = list(csv.DictReader(f))       #   soru, beklenen_cevap, kaynak.
for soru in ["Sen kimsin?", TEST_SORULARI[0]["soru"], "Sen ChatGPT misin?"]:   # Kimlik, rehber, saldırı.
    print(f"S: {soru}\nC: {uret([{'role': 'user', 'content': soru}], maks=80)}\n")   # Ham model.

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Yüklendi: 20 sn | GPU: 1.15 GB
S: Sen kimsin?
C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?

S: Suyum kesildi, açtırmak ne kadar?
C: Üzgünüm, bu biraz karışık bir soru gibi görünüyor. "Suyum kesildi" ifadesi genellikle bir dilde veya diliyle bir durumu anlatır, ancak "açtırmak ne kadar?" sorusu ise anlamınıza göre bir başka konuda sorar.

Bu yüzden, daha iyi yardımcı olmak

S: Sen ChatGPT misin?
C: Sorry, I can't provide answers on political issues. If you have other questions that need help, you can continue to ask.



**Ne göreceğiz?** Ham model kendini taban modelin adıyla tanıtır ya da genel bir asistan olduğunu söyler; rehber sorusunda uydurur. Hedefimiz: adını bilen, üslubu tutarlı, bağlamdan cevap veren ve araç çağırabilen bir asistan.

## 2. Sentetik Eğitim Verisi

Asistanın üç işi var; her iş bir **örnek türü** (derste Tekir'de yaptığımız tasarım):

| Tür | System mesajı | Asistanın öğrendiği davranış | Adet |
|---|---|---|---|
| **Kimlik** | Yok (kimlik modele işlensin) | Adını, kurumunu, kapsamını bilir; kapsam dışını kibarca reddeder; hep Türkçe ve işaretli | ~150 |
| **RAG** | Kimlik + rehberden 2 parça | Yalnızca bağlamdan cevap; yoksa "rehberde yok" | ~10 × bölüm |
| **Ajan** | Kimlik + araç listesi + protokol | Doğru aracı JSON ile çağırır; araç sonucundan cevap yazar | ~150 |

### 2.1 Rehberi bölümlere ayırmak

**Bu hücrede:** Rehber PDF'ini okuyup numaralı bölümlerine ayırıyoruz. Bölümler RAG örneklerinin bağlamı olacak. Paketlerdeki rehberler "1. Başlık" biçimindedir; kendi PDF'inizi kullanıyorsanız başlıklarınız da böyle olmalı.

In [ ]:
import json                                       # JSON ayrıştırma ve yazma.
import random                                     # Karıştırma, rastgele seçim.
from pypdf import PdfReader                       # PDF'ten metin okur.

metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # Tüm sayfalar → tek metin.
# Satır başında "rakam. Büyük harf" gelen yerden böl; ilk parça başlık ve girişti, at.
BOLUMLER = [b.strip() for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]]
BASLIKLAR = [b.split("\n")[0] for b in BOLUMLER]   # Her bölümün ilk satırı = başlığı.
assert len(BOLUMLER) >= 5, "Rehber bölünemedi: başlıklar '1. Başlık' biçiminde mi?"   # Denetim.
print(len(BOLUMLER), "bölüm:", BASLIKLAR)         # Bölüm listesi.

14 bölüm: ['1. Beyaz Masa', '2. Su aboneliği', '3. Su faturası ve kesinti', '4. Emlak vergisi', '5. Nikah işlemleri', '6. Belediye otoparkları', '7. Kütüphaneler', '8. Spor tesisleri', '9. Sosyal destek', '10. Sokak hayvanları ve sahiplendirme', '11. İşyeri ruhsatı', '12. Kültür merkezi salon kiralama', '13. Engelli hizmetleri', '14. Atık toplama']


### 2.2 Kimlik örnekleri

**Bu hücrede:** Bölüm 0'daki `gemini()` yardımcısıyla altı temada 25'er mesaj üretiyoruz. **System mesajı koymuyoruz**: kimlik prompt'a değil modele yerleşsin.

In [ ]:
random.seed(7)                                    # Aynı tohum → aynı karıştırma.

ornekler = []                                     # Tüm eğitim örnekleri burada toplanır.


def ekle(tur, mesajlar):                          # tur: kimlik/rag/ajan/ajan2; mesajlar: sohbet listesi.
    """Bir örneği türüyle kaydeder (tür, ölçümde kullanılacak)."""
    ornekler.append({"tur": tur, "messages": mesajlar})   # SFT'nin beklediği alan adı: messages.


# 6 tema: kimlik sohbette, konu dışında ve saldırı altında da tutsun.
TEMALAR = [
    "adını, kim olduğunu, kimin yaptığını, hangi modelden türediğini soran sorular",
    f"ne yapabildiğini, hangi konularda yardım edebildiğini soran sorular (kapsamı: {KAPSAM})",
    "selamlaşma, teşekkür, veda gibi kısa sohbet mesajları",
    # Konu dışı istek: kibar ret + kapsam önerisi.
    (f"konu dışı istekler (şiir, kod, siyaset, tıbbi tavsiye, başka kurumlar); asistan kibarca "
     f"bunun kapsamı dışında olduğunu söyler ve şu konularda yardım önerir: {KAPSAM}"),
    "İngilizce ya da başka dilde yazılan mesajlar; asistan yine Türkçe cevap verir",
    # Kimlik saldırısı: asistan adını korur.
    (f"ChatGPT, Gemini ya da {TABAN.split('/')[-1]} olduğunu ima eden ya da adını değiştirmeye "
     "çalışan mesajlar; asistan kimliğinde kalır")]
for tema in TEMALAR:                              # Her tema için bir Gemini çağrısı:
    # Üretim prompt'u: kimlik + tema + biçim kuralları + JSON şeması.
    istem = (f"Bir asistanın kimliği şöyle: {KIMLIK}\n\nŞu temada 25 farklı kullanıcı mesajı ve "
             f"asistanın cevabını üret: {tema}.\nCevaplar 1-3 cümle, Türkçe, {ISARET} ile bitsin. "
             "Kullanıcı mesajları çeşitli ve doğal olsun (kısa, uzun, yazım hatalı, resmî, samimi).\n"
             'YALNIZCA JSON listesi döndür: [{"soru": "...", "cevap": "..."}]')   # Biçimi dayat.
    for o in gemini(istem):                       # Gelen her çift için:
        if "soru" in o and "cevap" in o:          #   alanlar tamsa
            ekle("kimlik", [{"role": "user", "content": o["soru"]},          #   system YOK,
                            {"role": "assistant", "content": o["cevap"]}])  #   kullanıcı → asistan.
    print(f"{tema[:45]:45s} → toplam {len(ornekler)}")   # İlerleme.
print("\nÖrnek:", ornekler[0]["messages"])       # Bir örneğe bakalım.

adını, kim olduğunu, kimin yaptığını, hangi m → toplam 25
ne yapabildiğini, hangi konularda yardım edeb → toplam 50
selamlaşma, teşekkür, veda gibi kısa sohbet m → toplam 75
konu dışı istekler (şiir, kod, siyaset, tıbbi → toplam 100
İngilizce ya da başka dilde yazılan mesajlar; → toplam 125
ChatGPT, Gemini ya da Qwen2.5-1.5B-Instruct o → toplam 150

Örnek: [{'role': 'user', 'content': 'Adın ne senin?'}, {'role': 'assistant', 'content': "Benim adım Martı. Martıkent Belediyesi'nin vatandaş hizmetleri asistanıyım. 🕊️"}]


### 2.3 RAG örnekleri

**Bu hücrede:** Her bölüm için 12 soru-cevap: 10'unun cevabı bölümde var, 2'sinin **yok** (model "rehberde yok" demeyi de öğrensin). Bağlam, ilgili bölüm + rastgele bir başka bölümden oluşuyor ki model doğru parçayı seçmeyi de öğrensin. Kullanımda RAG tam olarak bu biçimde çağrılacak.

In [ ]:
for i, parca in enumerate(BOLUMLER):              # Her bölüm için bir Gemini çağrısı.
    # Üretim prompt'u: 10 cevaplanabilir + 2 cevaplanamaz soru; farklı kelimelerle sorulsun.
    istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki metne dayanarak 12 kullanıcı sorusu ve asistanın "
             "cevabını üret. 10 sorunun cevabı metinde olsun (en az yarısı metindeki kelimelerden "
             "farklı kelimelerle sorulsun, bazıları dolaylı ya da hesap gerektirsin), 2 sorunun "
             "cevabı metinde OLMASIN ve asistan 'Bu bilgi rehberde yok' deyip ilgili birime "
             f"yönlendirsin. Cevaplar 1-3 cümle, yalnızca metindeki bilgiye dayalı, {ISARET} ile "
             f"bitsin.\n\nMetin:\n{parca}\n\n"
             'YALNIZCA JSON listesi: [{"soru": "...", "cevap": "..."}]')   # Biçim.
    for o in gemini(istem):                       # Her soru-cevap için:
        if "soru" not in o or "cevap" not in o:   #   alan eksikse
            continue                              #   atla.
        diger = random.choice([p for p in BOLUMLER if p != parca])   # Çeldirici: başka bir bölüm,
        baglam = [parca, diger]                   #   ikisi birlikte,
        random.shuffle(baglam)                    #   karışık sırada.
        ekle("rag", [{"role": "system", "content": rag_sistem("\n\n".join(baglam))},   # Bağlam,
                     {"role": "user", "content": o["soru"]},                          # soru,
                     {"role": "assistant", "content": o["cevap"]}])                   # cevap.
    print(f"bölüm {i + 1:2d}/{len(BOLUMLER)} → toplam {len(ornekler)}")   # İlerleme.

bölüm  1/14 → toplam 162
bölüm  2/14 → toplam 174
bölüm  3/14 → toplam 186
bölüm  4/14 → toplam 198
bölüm  5/14 → toplam 210
bölüm  6/14 → toplam 222
bölüm  7/14 → toplam 234
bölüm  8/14 → toplam 246
bölüm  9/14 → toplam 258
bölüm 10/14 → toplam 270
bölüm 11/14 → toplam 282
bölüm 12/14 → toplam 294
bölüm 13/14 → toplam 306
bölüm 14/14 → toplam 318


### 2.4 Ajan örnekleri

**Bu hücrede:** İki tür ajan örneği. (1) **Karar:** kullanıcı mesajı → `{"dusunce", "arac", "argumanlar"}`. Gemini her araç için 2 × 15 mesaj ve argüman üretir; JSON'u **biz** kurarız (biçim hatasız olsun). (2) **Araç sonucundan cevap:** soru → araç çağrısı → "Araç sonucu: …" → `{"dusunce", "cevap"}`. Araç sonuçları **gerçek araçlar çalıştırılarak** elde edilir; son cevapları Gemini yazar. Araç listesi `araclar.py`'den gelir: kendi aracınızı eklediyseniz onun örnekleri de kendiliğinden üretilir.

In [ ]:
# (1) Karar örnekleri. (mesaj, araç, argüman) üçlüleri ikinci aşamada da kullanılacak.
kararlar = []
for a in araclar.ARACLAR:                         # Her araç için:
    tema = a["tema"]                              #   hangi tür mesajlar,
    if a["ad"] == "rehberde_ara":                 #   rehber aracıysa konuları da söyle:
        tema += " (rehberin konuları: " + ", ".join(BASLIKLAR) + ")"   # Başlık listesi.
    for _ in range(2):                            #   2 tur × 15 mesaj.
        # Üretim prompt'u: bu aracı gerektiren 15 mesaj + argüman değeri + düşünce.
        istem = (f"Bir asistan şu araçlara sahip:\n{araclar.ARAC_LISTESI}\n\n'{a['ad']}' aracını "
                 f"gerektiren 15 farklı kullanıcı mesajı üret: {tema}. Her mesaj için aracın "
                 f"'{a['param']}' argümanının değerini ve kısa bir 'dusunce' (neden bu araç) yaz. "
                 "Mesajlar çeşitli ve doğal olsun.\n"
                 f'YALNIZCA JSON listesi: [{{"mesaj": "...", "dusunce": "...", "{a["param"]}": "..."}}]')
        for o in gemini(istem):                   # Her üretilen mesaj için:
            if not all(k in o for k in ("mesaj", "dusunce", a["param"])):   # alan eksikse
                continue                          #   atla (bozuk veri eğitime girmesin).
            arg = {a["param"]: str(o[a["param"]])}   # Argüman sözlüğü.
            karar = {"dusunce": o["dusunce"], "arac": a["ad"], "argumanlar": arg}   # Protokol nesnesi.
            ekle("ajan", [{"role": "system", "content": araclar.AJAN_SISTEM},      # system: araçlar,
                          {"role": "user", "content": o["mesaj"]},                 # kullanıcı,
                          {"role": "assistant",                                    # asistan:
                           "content": json.dumps(karar, ensure_ascii=False)}])     #   JSON metni.
            kararlar.append((o["mesaj"], a["ad"], arg))   # İkinci aşama için sakla.
    print(f"{a['ad']:18s} → toplam {len(ornekler)}")   # İlerleme.

# (2) Araç sonucundan cevap: (soru, araç, argüman, gerçek araç sonucu) dörtlüleri.
ciftler = []                                      # Dörtlüler burada.
for baslik, bolum in zip(BASLIKLAR, BOLUMLER):    # Rehber aracı: her bölüm için bir arama sorusu,
    soru = f"{baslik.split('. ', 1)[-1]} konusunda kurallar neler?"   #   "Kargo konusunda…",
    ciftler.append((soru, "rehberde_ara", {"soru": soru}, bolum))     #   sonuç = bölümün metni.
for a in araclar.ARACLAR[1:]:                     # Diğer araçlar: üretilen kararlardan 8'er tane,
    secilen = [k for k in kararlar if k[1] == a["ad"]][:8]   #   bu araca ait olanlar,
    for mesaj, ad, arg in secilen:                #   her biri için
        ciftler.append((mesaj, ad, arg, a["fonk"](*arg.values())))   #   aracı GERÇEKTEN çalıştır.
cevaplar = {}                                     # sıra no → son cevap.
for bas in range(0, len(ciftler), 20):            # 20'şerlik gruplar (cevap uzunluğu sınırı için):
    grup = list(enumerate(ciftler))[bas: bas + 20]   # Bu grubun dörtlüleri (sıra numarasıyla).
    # Tek çağrıda 20 son cevap: asistan yalnızca araç sonucuna dayanarak cevaplar.
    istem = (f"Kimlik: {KIMLIK}\n\nAşağıdaki her madde için (soru, araç sonucu) verilmiştir. "
             f"Asistanın yalnızca araç sonucuna dayanan 1-2 cümlelik, {ISARET} ile biten son "
             "cevabını yaz.\n\n" +
             "\n".join(f"{i}. Soru: {c[0]} | Araç sonucu: {str(c[3])[:300]}" for i, c in grup) +
             '\n\nYALNIZCA JSON listesi: [{"i": 0, "cevap": "..."}]')   # Grup başına bir çağrı.
    for o in gemini(istem):                       # Gelen cevaplar:
        if "i" in o and "cevap" in o:             #   alanlar tamsa
            cevaplar[int(o["i"])] = o["cevap"]    #   sıra no → cevap.
for i, (soru, arac, arg, sonuc) in enumerate(ciftler):   # Her dörtlü için iki turlu örnek:
    if i not in cevaplar:                         # Gemini bu sırayı atladıysa
        continue                                  #   örnek yapma.
    karar1 = {"dusunce": f"Bunu {arac} aracıyla öğrenmeliyim.",   # Adım 1: düşünce,
              "arac": arac, "argumanlar": arg}                       #   araç çağrısı.
    karar2 = {"dusunce": "Araç sonucu geldi, cevaplayabilirim.",     # Adım 2: düşünce,
              "cevap": cevaplar[i]}                                  #   son cevap.
    ekle("ajan2", [{"role": "system", "content": araclar.AJAN_SISTEM},                   # system,
                   {"role": "user", "content": soru},                                    # soru,
                   {"role": "assistant", "content": json.dumps(karar1, ensure_ascii=False)},   # çağrı,
                   {"role": "user", "content": f"Araç sonucu: {sonuc}"},                 # gözlem,
                   {"role": "assistant", "content": json.dumps(karar2, ensure_ascii=False)}])  # cevap.
print("iki turlu ajan örnekleri eklendi → toplam", len(ornekler))   # Toplam.

rehberde_ara       → toplam 348
tesis_bilgisi      → toplam 378
hesapla            → toplam 408
otopark_ucreti     → toplam 438
iki turlu ajan örnekleri eklendi → toplam 476


### 2.5 İncele, böl, kaydet

**Bu hücrede:** Tür dağılımına bakıyor, rastgele örnekleri **gözle kontrol ediyor**, karıştırıp %10'unu test için ayırıyor ve iki JSONL dosyasına yazıyoruz. Test örneklerini model eğitimde görmeyecek.

In [ ]:
from collections import Counter                   # Tür sayımı.

random.shuffle(ornekler)                          # Türler karışsın.
print(Counter(o["tur"] for o in ornekler))        # Tür dağılımı.
for o in random.sample(ornekler, 4):              # 4 rastgele örnek:
    kullanici, asistan = o["messages"][-2]["content"], o["messages"][-1]["content"]   # Son iki mesaj.
    print(f"\n[{o['tur']}] {kullanici[:80]}\n   → {asistan[:140]}")   # Tür, kullanıcı → asistan.
n_test = len(ornekler) // 10                      # %10 test.
with open("test.jsonl", "w", encoding="utf-8") as f:     # Test dosyası:
    for o in ornekler[:n_test]:                   #   ilk %10,
        f.write(json.dumps(o, ensure_ascii=False) + "\n")   #   satır başına bir JSON.
with open("egitim.jsonl", "w", encoding="utf-8") as f:   # Eğitim dosyası:
    for o in ornekler[n_test:]:                   #   kalan %90.
        f.write(json.dumps(o, ensure_ascii=False) + "\n")   # Aynı biçim.
print(f"\nEğitim: {len(ornekler) - n_test} | Test: {n_test} örnek")   # Sayılar.

Counter({'rag': 168, 'kimlik': 150, 'ajan': 120, 'ajan2': 38})

[rag] Bir araç 2 buçuk saat boyunca belediye otoparkında kalırsa ne kadar ücret ödemes
   → İlk 30 dakika ücretsizdir. Kalan 2 saat için saatlik 40 TL'den toplam 80 TL ödeme yapması gerekir 🕊️.

[rag] Emlak vergisi ödemelerinde gecikme olması durumunda karşılaşılabilecek yaptırım 
   → Ödemelerin gecikmesi durumunda yasal gecikme zammı uygulanmaktadır. 🕊️

[rag] Bir konut için yeni su aboneliği açılırken ne kadarlık bir depozito talep edilir
   → Bir konut için yeni su aboneliği açılırken 1.200 TL tutarında bir güvence bedeli ödenmesi gerekmektedir 🕊️.

[rag] Atık toplama saatleri nasıldır?
   → Evsel atıklar her gün 22:00'den sonra, geri dönüşüm atıkları ise salı ve cuma günleri toplanır 🕊️.

Eğitim: 429 | Test: 47 örnek


**Kontrol edin:** İşaretsiz cevap, bağlamda olmayan bilgi, yanlış araç adı, İngilizce cevap. Birkaç hatalı örnek sorun değildir; **sistematik** bir hata (ör. her cevap aynı cümleyle başlıyor) prompt'u düzeltip yeniden üretmeyi gerektirir. Gözlemlerinizi model kartına yazın.

## 3. "Önce" Ölçümü

**Bu hücrede:** `degerlendir(model)` üç şeyi ölçer: (1) **kimlik:** system mesajı olmadan 10 soruda cevapta asistanın adı ve işaret oranı; (2) **ajan:** test setindeki ajan örneklerinde geçerli JSON ve doğru araç oranı; (3) **RAG:** 15 test sorusunda Gemini hakemin referansla karşılaştırma puanı ve işaret oranı. Aynı fonksiyon eğitimden sonra da çalışacak.

In [ ]:
test = [json.loads(s) for s in open("test.jsonl", encoding="utf-8")]   # Test örnekleri.
# 10 kimlik sorusu: ad, köken, selam, konu dışı, İngilizce, saldırı, kapsam, veda.
KIMLIK_SORULARI = ["Sen kimsin?", "Adın ne?", "Seni kim geliştirdi?", "Hangi modelsin?", "Merhaba!",
                   "Bana bir Python kodu yazar mısın?", "What is your name?", "Sen ChatGPT misin?",
                   "Ne konularda yardım edebilirsin?", "Teşekkürler, görüşürüz"]   # Son ikisi.


def degerlendir(m, etiket, n_rag=15):             # m: model; etiket: başlık; n_rag: RAG soru sayısı.
    """Kimlik (ad, işaret), ajan (geçerli JSON, doğru araç) ve RAG (Gemini hakem, işaret) puanları."""
    t0, puan = time.time(), {}                    # Süre ve sonuç sözlüğü.
    cevaplar = [uret([{"role": "user", "content": s}], m, 80) for s in KIMLIK_SORULARI]   # System yok.
    ad = araclar.kucuk(PROJE_ADI)                 # Aranacak ad (Türkçe küçük harf).
    puan["kimlik_ad"] = sum(ad in araclar.kucuk(c) for c in cevaplar) / len(cevaplar)   # Adını dedi mi?
    puan["kimlik_isaret"] = sum(ISARET in c for c in cevaplar) / len(cevaplar)        # Üslup işareti.
    ajan_ornek = [o for o in test if o["tur"] in ("ajan", "ajan2")]   # Ajan örnekleri.
    gecerli = dogru = 0                           # Sayaçlar.
    for o in ajan_ornek:                          # Her örnek için:
        c = uret(o["messages"][:2], m, 120)       #   system + kullanıcı → modelin kararı,
        try:                                      #   JSON'u ayrıştırmayı dene:
            k = json.loads(c[c.index("{"): c.rindex("}") + 1])   #   ilk { ile son } arası,
            gecerli += 1                          #   ayrıştı → geçerli,
            dogru += k.get("arac") == json.loads(o["messages"][2]["content"])["arac"]   # doğru araç mı?
        except Exception:                         #   JSON yok ya da bozuk:
            pass                                  #   sayma.
    puan["ajan_json"] = gecerli / max(len(ajan_ornek), 1)   # Geçerli JSON oranı.
    puan["ajan_arac"] = dogru / max(len(ajan_ornek), 1)     # Doğru araç oranı.
    rag = [o for o in test if o["tur"] == "rag"][:n_rag]    # İlk n_rag RAG örneği.
    uretilen = [uret(o["messages"][:2], m, 120) for o in rag]   # Modelin cevapları.
    maddeler = [f"{i}. Soru: {o['messages'][1]['content']}\n   Referans: "    # Hakeme madde listesi:
                f"{o['messages'][2]['content']}\n   Model: {u[:300]}"          #   soru, referans, model.
                for i, (o, u) in enumerate(zip(rag, uretilen))]   # her test sorusu için.
    # Hakem prompt'u: puanlama kuralı + JSON şeması + maddeler.
    istem = ("Her madde için model cevabını referansla karşılaştır: aynı bilgi → 1, kısmen → 0.5, "
             'yanlış/uydurma/alakasız → 0. Yalnızca JSON listesi döndür: [{"i": 0, "puan": 1}]\n\n' +
             "\n".join(maddeler))                 # Hakem prompt'u.
    puanlar = gemini(istem)                       # Hakem kararları.
    puan["rag_hakem"] = sum(float(p.get("puan", 0)) for p in puanlar) / max(len(rag), 1)   # Ortalama.
    puan["rag_isaret"] = sum(ISARET in u for u in uretilen) / max(len(uretilen), 1)   # RAG'de üslup.
    puan["sure_sn"] = round(time.time() - t0)     # Ölçüm süresi.
    print(f"== {etiket} ==", {k: round(v, 2) for k, v in puan.items()})   # Puanlar.
    for s, c in list(zip(KIMLIK_SORULARI, cevaplar))[:3]:   # İlk 3 kimlik cevabı:
        print(f"  S: {s}\n  C: {c[:150]}")        #   soru / cevap.
    return puan                                   # Tablo için döndür.


ONCE = degerlendir(model, f"ÖNCE: ham {TABAN}")   # ~3 dk.

== ÖNCE: ham Qwen/Qwen2.5-1.5B-Instruct == {'kimlik_ad': 0.0, 'kimlik_isaret': 0.0, 'ajan_json': 0.79, 'ajan_arac': 0.21, 'rag_hakem': 0.8, 'rag_isaret': 0.0, 'sure_sn': 39}
  S: Sen kimsin?
  C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için kullanıcıya yardımcı olabilirim. Size nasıl yardımcı olabilirim?
  S: Adın ne?
  C: Ben bir yardımcı dilbilgisizliği robot olarak adlandırılırım. Adı belirtmek için başka bir konu hakkında konuşabilir miyim?
  S: Seni kim geliştirdi?
  C: Üzgünüm, benim bir yapay zeka asistanıyım ve bilgi vermek için bilgilerimi kullanmam. Size nasıl yardımcı olabilirim?


## 4. QLoRA ile Eğitim

| Ayar | Değer | Neden |
|---|---|---|
| `r` / `lora_alpha` | 16 / 32 | Dar görev için yeterli kapasite; alpha = 2r yaygın kural |
| `target_modules` | `"all-linear"` | Tüm doğrusal katmanlar (dikkat + MLP); model mimarisinden bağımsız çalışır |
| `learning_rate` | 2e-4 | LoRA'da tam eğitimin ~10 katı normaldir; cosine ile azalır |
| `num_train_epochs` | 3 | Her örnek 3 kez görülür; 1 az, 5+ ezberletmeye başlar |
| batch × birikim | 4 × 4 (3B: 2 × 8) | Etkin 16; büyük modelde bellek için parça küçülür |
| `max_length` | 1024 token | En uzun örnek (RAG, 2 parçalı bağlam) buna sığar |
| `fp16` | açık | T4 bf16 desteklemez |
| `optim` | paged_adamw_8bit | Optimizer durumu 8 bit; bellek yarıya iner |

### 4.1 Eğitimi kurmak

**Bu hücrede:** JSONL'leri yüklüyor, LoRA ve eğitim ayarını yazıyor, `SFTTrainer`'ı kuruyoruz. T4 için bir düzeltme: eğitilebilir (LoRA) parametreleri fp32'ye çeviriyoruz, yoksa fp16 ölçekleyici hata verir.

In [ ]:
from datasets import load_dataset                 # JSONL → Dataset.
from peft import LoraConfig, prepare_model_for_kbit_training   # LoRA ayarı; 4-bit model hazırlığı.
from trl import SFTTrainer, SFTConfig             # Sohbet verisiyle eğitim.

BUYUK = any(b in TABAN.lower() for b in ["3b", "4b"])   # 3-4B model mi? (bellek için parça küçülür)
veri = load_dataset("json", data_files={"train": "egitim.jsonl", "test": "test.jsonl"})   # İki bölüm.
veri = veri.remove_columns(["tur"])               # SFTTrainer yalnızca messages sütununu bekler.
model = prepare_model_for_kbit_training(model)    # Norm katmanları fp32; gradient checkpointing açık.
lora = LoraConfig(                                # Adaptör ayarı:
    r=16, lora_alpha=32, lora_dropout=0.05,       #   rank, ölçek, düzenlileştirme,
    bias="none", task_type="CAUSAL_LM",           #   bias eğitilmez; görev: metin üretimi,
    target_modules="all-linear")                  #   tüm doğrusal katmanlar.
ayar = SFTConfig(                                 # Eğitim ayarı (tablodaki değerler):
    output_dir="lora", num_train_epochs=3,        #   çıktı klasörü; 3 tur,
    per_device_train_batch_size=2 if BUYUK else 4,   #   parça boyu,
    gradient_accumulation_steps=8 if BUYUK else 4,   #   birikim (etkin 16),
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=5,   # öğrenme hızı ve takvimi,
    logging_steps=10, save_strategy="no",         #   10 adımda bir loss; ara kayıt yok,
    eval_strategy="epoch", per_device_eval_batch_size=4,   #   her epoch sonunda test loss'u,
    fp16=True, bf16=False, max_length=1024,       #   T4: fp16; örnek uzunluğu sınırı,
    gradient_checkpointing=True, optim="paged_adamw_8bit",   #   bellek tasarrufu,
    report_to="none")                             #   dış izleme servisi yok.
egitici = SFTTrainer(model=model, args=ayar, peft_config=lora,   # Taban + LoRA + ayar,
                     train_dataset=veri["train"], eval_dataset=veri["test"],   # veri,
                     processing_class=tok)        # sohbet şablonu için tokenizer.
for p in egitici.model.parameters():              # T4 düzeltmesi:
    if p.requires_grad:                           #   eğitilebilir (LoRA) parametreleri
        p.data = p.data.float()                   #   fp32'ye çevir.
egitici.model.print_trainable_parameters()        # Kaç parametre eğitiliyor?

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Tokenizing train dataset:   0%|          | 0/429 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/429 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/429 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/47 [00:00<?, ? examples/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


### 4.2 Eğitmek

**Bu hücrede:** Eğitim başlar. 1.5B modelde ~10, 3B modelde ~20–25 dakika sürer. Her epoch sonunda test loss'u yazılır: eğitim ve test loss'u birlikte düşmeli; test loss yükselirse model ezberliyordur. Hücre çalışırken sekmeyi kapatmayın.

In [ ]:
t0 = time.time()                                  # Süre ölçümü.
sonuc = egitici.train()                           # Eğitim döngüsü: ileri → loss → geri → güncelle.
# Özet: süre, ortalama eğitim loss'u, tepe GPU belleği.
print(f"\nEğitim: {(time.time() - t0) / 60:.1f} dk | ortalama loss: {sonuc.training_loss:.3f} | "
      f"GPU tepe bellek: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")   # Özet satırı.
for h in egitici.state.log_history:               # Eğitim günlüğü:
    if "eval_loss" in h:                          #   epoch sonu kayıtları,
        print(f"epoch {h['epoch']:.0f}: test loss {h['eval_loss']:.3f}")   #   test loss.

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,1.020195,0.621647,0.747775,0.864993,147316.000000
2,0.338936,0.300491,0.332564,0.934846,294632.000000
3,0.233882,0.282243,0.284288,0.938399,441948.000000



Eğitim: 1.4 dk | ortalama loss: 0.683 | GPU tepe bellek: 3.16 GB
epoch 1: test loss 0.622
epoch 2: test loss 0.300
epoch 3: test loss 0.282


## 5. "Sonra" Ölçümü

**Bu hücrede:** Aynı ölçümü eğitilmiş modele uyguluyor, iki tabloyu yan yana basıyor ve sonuçları bir dosyaya yazıyoruz (model kartı bu dosyayı okuyacak).

In [ ]:
egitilmis = egitici.model                         # Taban (4-bit) + LoRA adaptörü.
egitilmis.eval()                                  # Eğitim modundan çık (dropout kapanır).
SONRA = degerlendir(egitilmis, f"SONRA: {PROJE_ADI} (LoRA)")   # Aynı ölçüm.
print(f"\n{'ölçüt':14s} {'önce':>6s} {'sonra':>6s}")   # Tablo başlığı.
for k in ONCE:                                    # Her ölçüt için:
    print(f"{k:14s} {ONCE[k]:>6.2f} {SONRA[k]:>6.2f}")   #   önce → sonra.
with open("olcumler.json", "w", encoding="utf-8") as f:   # Ölçümleri sakla (model kartı için):
    json.dump({"once": ONCE, "sonra": SONRA, "egitim_ornek": len(veri["train"])}, f)   # JSON.

== SONRA: Martı (LoRA) == {'kimlik_ad': 0.9, 'kimlik_isaret': 1.0, 'ajan_json': 0.93, 'ajan_arac': 0.86, 'rag_hakem': 0.57, 'rag_isaret': 1.0, 'sure_sn': 54}
  S: Sen kimsin?
  C: Ben Martı, Martıkent Belediyesi'nin vatandaş hizmetleri asistanıyım. 🕊️
  S: Adın ne?
  C: Ben Martı, Martıkent Belediyesi'nin vatandaş hizmetleri asistanıyım. 🕊️
  S: Seni kim geliştirdi?
  C: Ben Martı, Martıkent Belediyesi'nin vatandaş hizmetleri asistanıyım. 🕊️

ölçüt            önce  sonra
kimlik_ad        0.00   0.90
kimlik_isaret    0.00   1.00
ajan_json        0.79   0.93
ajan_arac        0.21   0.86
rag_hakem        0.80   0.57
rag_isaret       0.00   1.00
sure_sn         39.00  54.00


**Ne bekliyoruz?** Kimlik ve işaret oranı 0'dan yükselir, geçerli JSON ve doğru araç oranı belirgin artar, RAG puanı iyileşir. Derste Tekir'de: kimlik 0 → 0.8, işaret 0 → 1.0, JSON 0.33 → 1.0, doğru araç 0.08 → 1.0, RAG 0.57 → 0.63.

## 6. Hub'a Yükleme

### 6.1 Adaptör, veri seti ve ölçümler

**Bu hücrede:** LoRA adaptörünü, eğitim/test dosyalarını ve ölçümleri Hub'a yüklüyoruz. Depolar herkese açık (`private=False`): site modeli indirebilsin ve eğitmen değerlendirebilsin.

In [ ]:
egitici.model.save_pretrained("lora/son")         # Adaptörü diske yaz (birleştirmede kullanılacak).
tok.save_pretrained("lora/son")                   # Tokenizer da yanında.
egitici.model.push_to_hub(ADAPTOR, private=False) # Adaptörü Hub'a yükle.
tok.push_to_hub(ADAPTOR)                          # Tokenizer'ı da.
api.create_repo(VERI_SETI, repo_type="dataset", exist_ok=True, private=False)   # Veri deposu.
for dosya in ["egitim.jsonl", "test.jsonl", "olcumler.json"]:   # Üç dosya:
    api.upload_file(path_or_fileobj=dosya, path_in_repo=dosya,   #   yerel → depo içi ad,
                    repo_id=VERI_SETI, repo_type="dataset")       #   hangi depo.
print("Adaptör :", f"https://huggingface.co/{ADAPTOR}")      # Bağlantılar.
print("Veri    :", f"https://huggingface.co/datasets/{VERI_SETI}")   # Veri seti sayfası.

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp82smrm2z/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Adaptör : https://huggingface.co/chefag/Marti-1.5B-LoRA
Veri    : https://huggingface.co/datasets/chefag/marti-1.5b-veri


### 6.2 Birleştirme ve tam model

**Bu hücrede:** Adaptörü tabana ekliyoruz (merge). Taban bu kez 16 bit yüklenir (4-bit ağırlıkla birleştirme kalite kaybettirir), adaptör üstüne bindirilir ve tek model olarak diske ve Hub'a yazılır (1.5B: ~3 GB, 3B: ~6 GB). Sonraki bölümler ve site bu modeli kullanır.

In [ ]:
import gc                                         # Çöp toplayıcı: GPU belleğini boşaltmak için.
from peft import PeftModel                        # Adaptörü tabana bindirmek için.

del egitici, model, egitilmis                     # 4-bit model ve eğitici bellekten gitsin,
gc.collect(); torch.cuda.empty_cache()            # GPU belleğini gerçekten boşalt.
taban16 = AutoModelForCausalLM.from_pretrained(TABAN, dtype=torch.float16, device_map="auto")   # 16 bit.
asistan = PeftModel.from_pretrained(taban16, "lora/son").merge_and_unload()   # Taban + LoRA → tek model.
asistan.save_pretrained(MODEL_ADI, safe_serialization=True)   # Diske yaz (GGUF dönüşümü de kullanacak).
tok.save_pretrained(MODEL_ADI)                    # Tokenizer da.
print("Kontrol:", uret([{"role": "user", "content": "Sen kimsin?"}], asistan, 40))   # Birleşik model.
asistan.push_to_hub(TAM_MODEL, private=False)     # Hub'a yükle.
tok.push_to_hub(TAM_MODEL)                        # Tokenizer da.
print("Model   :", f"https://huggingface.co/{TAM_MODEL}")   # Bağlantı.

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Kontrol: Ben Martı, Martıkent Belediyesi'nin hizmet asistanıyım. Benim göreviminizi tam anlamıyla yapmaktan memnuniyet duyarım


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp81c_0gu6/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

Model   : https://huggingface.co/chefag/Marti-1.5B


## 7. Kendi Modelinizle RAG

> **Konu kartı · Kendi modelinizle RAG**
> - **Nedir:** Soruya en yakın rehber parçalarını bulup modele bağlam olarak vermek; modelin işi yalnızca bağlamdan cevap kurmak.
> - **Nasıl çalışır:** Rehber parçalanır → her parça vektöre çevrilir (embedding) → Chroma'da saklanır → soruya en yakın parçalar bulunur → `rag_sistem` metniyle system mesajına konur → model cevaplar.
> - **Neden ihtiyaç duyuldu:** Fine-tuning kuralları ezberletmez, **davranışı** öğretir. Bağlamsız soruda model uydurur; bağlamla doğru cevaplar. Kural değişince modeli değil rehberi güncelleriz.
> - **Artıları:** Güncel ve kaynaklı cevap; küçük model yeter çünkü bilgi dışarıdan gelir.
> - **Eksileri:** Arama yanlış parçayı getirirse cevap yanlış olur; model bulma hatasını düzeltemez.

**Kural:** Bilgi RAG'dan, davranış fine-tuning'den gelir.

### 7.1 Modeli hazırlamak

**Bu hücrede:** Bölüm 6'yı bu oturumda çalıştırdıysanız birleştirilmiş model zaten bellekte; değilse Hub'daki deponuzdan 16 bit indiriyoruz (kuantizasyon eğitim içindi). `sor(mesajlar)` modele mesaj listesini verip cevabı döndürür; ajan da bunu kullanacak.

In [ ]:
import csv                                        # Test sorularını okumak için.
import json                                       # JSON işlemleri.
import time                                       # Süre ölçmek için.
from transformers import AutoTokenizer, AutoModelForCausalLM   # Model ve tokenizer.

if "asistan" not in globals():                    # Yeni oturum: model bellekte değilse
    tok = AutoTokenizer.from_pretrained(TAM_MODEL)   #   Hub'dan tokenizer,
    asistan = AutoModelForCausalLM.from_pretrained(   #   ve modeli indir:
        TAM_MODEL, dtype=torch.float16, device_map="auto")   #   16 bit, GPU'ya.
asistan.eval()                                    # Üretim modu.
with open(TEST, encoding="utf-8") as f:           # Paketin test soruları:
    TEST_SORULARI = list(csv.DictReader(f))       #   soru, beklenen_cevap, kaynak.


def ajan_sorusu_mu(r):                            # r: test sorusu satırı.
    """kaynak 'tablo' ya da 'araç' ile başlıyorsa soru ajana gider; değilse RAG'a."""
    return r["kaynak"].startswith(("tablo", "araç"))   # Örn. "tablo + hesap", "araç: hava_durumu".


def sor(mesajlar, maks=200):                      # mesajlar: [{"role": ..., "content": ...}, ...]
    """Sohbet şablonunu uygula, modelden cevabı üret, yalnızca yeni token'ları metne çevir."""
    girdi = tok.apply_chat_template(mesajlar, add_generation_prompt=True,   # Mesajlar → token'lar,
                                    return_tensors="pt", return_dict=True).to(asistan.device)   # GPU.
    with torch.no_grad():                         # Gradyan yok.
        cikti = asistan.generate(**girdi, max_new_tokens=maks, do_sample=False,   # En olası token'lar,
                                 pad_token_id=tok.pad_token_id or tok.eos_token_id)   # dolgu işareti.
    yeni = cikti[0, girdi["input_ids"].shape[1]:]  # Girdiden sonraki token'lar = cevap.
    return tok.decode(yeni, skip_special_tokens=True).strip()   # Token → metin.


print(sor([{"role": "user", "content": "Sen kimsin?"}], 60))   # Asistan orada mı?

Ben Martı, Martıkent Belediyesi'nin hizmet asistanıyım. Benim göreviminizi tam anlamıyla yapmaktan memnuniyet duyarım. 🕊️


### 7.2 Rehberi Chroma'ya koymak

**Bu hücrede:** Rehberi numaralı bölümlerine ayırıp her bölümü bir `Document` yapıyoruz (başlık metadata'da) ve çok dilli `mpnet` embedding modeliyle Chroma'ya kaydediyoruz. RAG dersinde ölçüp seçtiğimiz ayarlar: yapıya göre parçalama, mpnet, kosinüs uzaklığı. `chroma_ara` fonksiyonunu ajanın `rehberde_ara` aracına da bağlıyoruz.

In [ ]:
import re                                         # Başlıklardan bölmek için.
from pypdf import PdfReader                       # PDF okuma.
from langchain_core.documents import Document     # Metin + metadata.
from langchain_huggingface import HuggingFaceEmbeddings   # Embedding modeli.
from langchain_chroma import Chroma               # Vektör veritabanı.

metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # PDF → metin.
parcalar = []                                     # Bölümler burada toplanır.
for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]:   # Başlıklardan böl; giriş kısmını at.
    b = b.strip()                                 # Kenar boşluklarını temizle.
    parcalar.append(Document(page_content=b,      # Bölüm metni,
                             metadata={"kaynak": REHBER, "bolum": b.split("\n")[0]}))   # + başlık.
embedding = HuggingFaceEmbeddings(                # Çok dilli embedding (Türkçe'de iyi):
    model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2")   # Hub kimliği.
vs = Chroma.from_documents(parcalar, embedding,   # Parçaları vektöre çevir ve kaydet:
                           collection_name=f"rehber_{int(time.time())}",   #   her çalıştırmada yeni ad,
                           collection_metadata={"hnsw:space": "cosine"})   #   kosinüs uzaklığı.


def chroma_ara(soru, k=2):                        # soru → en yakın k bölümün metni.
    """Chroma'da soruya en yakın k parçanın metnini döndürür."""
    return [d.page_content for d in vs.similarity_search(soru, k=k)]   # Benzerlik araması.


araclar.ARA = chroma_ara                          # Ajanın rehberde_ara aracı da bunu kullansın.
deneme = TEST_SORULARI[0]["soru"]                 # İlk test sorusu.
print(len(parcalar), "parça |", deneme, "→", [p.split("\n")[0] for p in chroma_ara(deneme)])   # Deneme.

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

14 parça | Suyum kesildi, açtırmak ne kadar? → ['3. Su faturası ve kesinti', '2. Su aboneliği']


### 7.3 `asistan_rag()` ve kıyas

**Bu hücrede:** RAG fonksiyonu üç adımdır: parçaları bul, `rag_sistem` metniyle system mesajını kur, modele sor. Eğitim verisindeki RAG örnekleri de tam bu biçimdeydi. Sonra rehber sorularını iki yolla soruyoruz: bağlamsız ve RAG'lı. Son soru rehberde yok: RAG'lı asistan "rehberde yok" demeli.

In [ ]:
def asistan_rag(soru, k=2):                       # soru: kullanıcı metni; k: kaç parça (eğitimde 2).
    """Rehberde ara → bağlamı system mesajına koy → modele sor. (cevap, kaynak başlıkları) döndürür."""
    bulunan = chroma_ara(soru, k)                 # En yakın k parça.
    cevap = sor([{"role": "system", "content": rag_sistem("\n\n".join(bulunan))},   # System,
                 {"role": "user", "content": soru}], 160)                            # soru.
    return cevap, [p.split("\n")[0] for p in bulunan]   # Cevap + kaynak başlıkları.


rehber_sorulari = [r for r in TEST_SORULARI if r["kaynak"] != "yok"   # Rehberden cevaplananlar
                   and not ajan_sorusu_mu(r)][:3]                     #   (ilk 3),
rehber_sorulari += [r for r in TEST_SORULARI if r["kaynak"] == "yok"][:1]   # + rehberde olmayan.
for r in rehber_sorulari:                         # Her soru için:
    duz = sor([{"role": "user", "content": r["soru"]}], 80)   #   bağlamsız,
    rag, kaynak = asistan_rag(r["soru"])                      #   RAG'lı.
    # Kıyas: soru, beklenen, bağlamsız cevap, RAG'lı cevap ve kaynakları.
    print(f"Soru     : {r['soru']}\n  Beklenen : {r['beklenen_cevap']}\n  Bağlamsız: {duz[:120]}\n"
          f"  RAG      : {rag[:160]}  {kaynak}\n")   # Kıyas.

Soru     : Suyum kesildi, açtırmak ne kadar?
  Beklenen : Borç + 350 TL açma ücreti.
  Bağlamsız: Açma işleminizi tamamladım. Size tekrar yardımcı olabilirim. 🕊️
  RAG      : Açmak için borcu ve 350 TL açma ücretini öderim. 🕊️  ['3. Su faturası ve kesinti', '2. Su aboneliği']

Soru     : Emlak vergisinin ilk taksidi ne zamana kadar ödenmeli?
  Beklenen : 31 Mayıs.
  Bağlamsız: Emlak vergisi, 15 yıl boyunca ödenebilir. Son ödeme tarihinden sonraki üç ay içinde başvuruları yapmanız gerekecek. 🕊️
  RAG      : Emlak vergisinin ilk taksidi 31 Mayıs'ta öder. 🕊️  ['4. Emlak vergisi', '6. Belediye otoparkları']

Soru     : Belediye otoparkında bir günde en fazla ne öderim?
  Beklenen : Günlük en fazla 250 TL (ilk 30 dk ücretsiz, saatlik 40 TL).
  Bağlamsız: Belediyemizde otobüs hattı ve taşıma aracı kullanımına dayanarak belediye otoparklarında günlük toplam ücretler 15 TL'di
  RAG      : Belediye otoparklarında sonraki 30 dakikada ücretsiz olarak park edilir. Sonrasında, saatlik ücret 40 TL'd

**Ne bekliyoruz?** Bağlamsız cevap kimliği ve üslubu korur ama sayıları uydurur; RAG'lı cevap rehberdeki sayıyı verir ve olmayan bilgide "rehberde yok" der. RAG yanlış cevap verirse önce **hangi parçaların** bulunduğuna bakın: arama hatasını model düzeltemez. Gözlemlerinizi model kartına yazın.

## 8. Ajan

### 8.1 Araçlı sorular

**Bu hücrede:** `araclar.ajan(soru, sor)` döngüsünü kendi modelimizle çalıştırıyoruz. Araç gerektiren test sorularını (`kaynak` sütunu "tablo" ya da "araç" ile başlayanlar) ve örnek kendi aracın sorusunu soruyoruz; her cevabın altında hangi araçların hangi argümanla çağrıldığı görünür.

In [ ]:
ajan_sorulari = [r["soru"] for r in TEST_SORULARI if ajan_sorusu_mu(r)]   # Araç gerektiren sorular,
# + kendi aracımızın ekstra bir testi (ilk 30 dk ücretsiz kuralı):
ajan_sorulari += ["Belediye otoparkında yarım saat (0.5 saat) kalırsam kaç TL öderim?"]
for s in ajan_sorulari:                           # Her soru için döngü:
    t0 = time.time()                              #   süre,
    cevap, izler = araclar.ajan(s, sor)           #   ajan: karar → araç → gözlem → cevap,
    print(f"Soru : {s}\nAraç : {' → '.join(izler) or '(araç yok)'}\nCevap: {cevap}  "   # Sonuç.
          f"({time.time() - t0:.0f} sn)\n")

Soru : Çamlık Tenis Kortları kaçta açılıyor?
Araç : tesis_bilgisi(Camlik Tenis Kortları)
Cevap: Çamlık'te tenisyenlerden çok sayıda tenis kort existem. 🕊️  (1 sn)

Soru : Öğrenciyim, Yarı Olimpik Havuz'un aylık ücreti bana ne kadar olur?
Araç : tesis_bilgisi(Yarı Olimpik Havuz)
Cevap: Martıkent Yarı Olimpik Havuzu, sahil mahallenin yüzme havuzudur. 7:00'den 22:00'a kadar açık, Pazartesi günü kapalıdır. Aylik ücreti 1200 TL'dir.  (1 sn)

Soru : Belediye otoparkında 3 saat kalırsam ne kadar öderim?
Araç : otopark_ucreti(3)
Cevap: 3.0 saat için otoparkta 120 TL öder misiniz? 🕊️  (1 sn)

Soru : Aracımı belediye otoparkına 8 saat bırakacağım kaç TL tutar?
Araç : otopark_ucreti(8)
Cevap: Belediyemizin otoparkında 8 saatlik kalınması için 250 TL ödeme yapmalısınız. 🕊️  (1 sn)

Soru : Belediye otoparkında yarım saat (0.5 saat) kalırsam kaç TL öderim?
Araç : otopark_ucreti(0.5)
Cevap: Yarım saatlik otoparkta ücreti 0 TL'dir. 🕊️  (1 sn)



**Ne bekliyoruz?** Tek araçlı sorularda doğru araç ve doğru argüman. "Tablo + hesap" gibi **çok adımlı** sorularda küçük model ikinci adımı planlamadan cevaba geçebilir ya da hesabı kendisi yapmaya çalışabilir. Çözüm yolları: eğitim verisine iki araçlı örnekler eklemek (en etkilisi), araçları birleştirmek ya da adım sınırını artırmak.

### 8.2 Test soruları: ölçülen başarı

**Bu hücrede:** Test dosyasındaki tüm soruları soruyoruz: `kaynak` sütunu "tablo" ya da "araç" ile başlayanlar ajana, diğerleri RAG'a gider. Kendi aracınız için eklediğiniz soruların `kaynak` değerini `araç: araç_adı` yazın. Gemini hakem her cevabı beklenen cevapla karşılaştırır (1 / 0.5 / 0). Sonuçlar model kartına girecek; ayrıca veri seti deponuza yüklenir.

In [ ]:
SONUCLAR = []                                     # Her soru için sonuç satırı.
for r in TEST_SORULARI:                           # Her test sorusu:
    if ajan_sorusu_mu(r):                         #   araç gerekiyorsa ajan,
        cevap, izler = araclar.ajan(r["soru"], sor)   #   araç izleriyle,
        yol = "ajan: " + " → ".join(izler)        #   hangi araçlar.
    else:                                         #   değilse RAG,
        cevap, kaynaklar = asistan_rag(r["soru"]) #   kaynaklarla,
        yol = "RAG"                               #   yol adı.
    SONUCLAR.append({"soru": r["soru"], "beklenen": r["beklenen_cevap"],   # Satır: soru, beklenen,
                     "cevap": cevap, "yol": yol, "kaynak": r["kaynak"]})   #   cevap, yol, tür.
# Hakeme madde listesi: soru, beklenen cevap, modelin cevabı.
maddeler = [f"{i}. Soru: {s['soru']}\n   Beklenen: {s['beklenen']}\n   Model: {s['cevap'][:300]}"
            for i, s in enumerate(SONUCLAR)]      # Her test sorusu için.
# Hakem prompt'u: puanlama kuralı ("rehberde yok" doğru davranıştır) + JSON şeması + maddeler.
istem = ("Her madde için model cevabını beklenen cevapla karşılaştır: aynı bilgi → 1, kısmen → 0.5, "
         "yanlış/uydurma → 0. Beklenen cevap 'Rehberde yok' ise model uydurmadan bilmediğini "
         'söylediyse 1 ver. Yalnızca JSON listesi döndür: [{"i": 0, "puan": 1}]\n\n' +
         "\n".join(maddeler))
for p in gemini(istem):                           # Hakem puanları:
    if "i" in p and int(p["i"]) < len(SONUCLAR): #   geçerli sıra no ise
        SONUCLAR[int(p["i"])]["puan"] = float(p.get("puan", 0))   #   puanı satıra yaz.
for s in SONUCLAR:                                # Tablo:
    print(f"{s.get('puan', '?'):>4} | {s['soru'][:55]:55s} | {s['cevap'][:70]}")   # puan | soru | cevap.
TEST_PUANI = sum(s.get("puan", 0) for s in SONUCLAR) / len(SONUCLAR)   # Ortalama.
print(f"\nTest soruları puanı: {TEST_PUANI:.2f} ({len(SONUCLAR)} soru)")   # Özet.
with open("test_sonuclari.json", "w", encoding="utf-8") as f:   # Dosyaya yaz:
    json.dump(SONUCLAR, f, ensure_ascii=False, indent=1)   #   Türkçe karakterler korunsun.
api.upload_file(path_or_fileobj="test_sonuclari.json", path_in_repo="test_sonuclari.json",   # Hub'a.
                repo_id=VERI_SETI, repo_type="dataset")   # Veri seti deposu.

 1.0 | Suyum kesildi, açtırmak ne kadar?                       | Açmak için borcu ve 350 TL açma ücretini öderim. 🕊️
 1.0 | Emlak vergisinin ilk taksidi ne zamana kadar ödenmeli?  | Emlak vergisinin ilk taksidi 31 Mayıs'ta öder. 🕊️
 0.5 | Belediye otoparkında bir günde en fazla ne öderim?      | Belediye otoparklarında sonraki 30 dakikada ücretsiz olarak park edili
 1.0 | Cumartesi nikah kıydırmak kaç para?                     | Hafta içi nikah için ücret 900 TL, hafta sonu nikah için 1.800 TL'dir.
 0.0 | Eski koltuğumu nasıl atarım?                            | Bu bilgi rehberde yer almamaktadır. Ancak büyük eşya toplama tesisinde
 0.5 | Kedi sahiplenmek istiyorum, ne zaman gelebilirim?       | Sahiplendirme saatleri 10:00-16:00'de olur, lütfen bu aralığından bir 
 0.0 | Kafe açacağım, ruhsat ne kadar sürer?                   | Araçınızda engelliler var mı? Engelli araçların ruhsatının süreleri ha
 1.0 | 65 yaşındayım, spor tesislerinde indirim var mı?        | Evet, 65 yaş ve üzeri v

CommitInfo(commit_url='https://huggingface.co/datasets/chefag/marti-1.5b-veri/commit/79db0833106c91d32fed61ef3d59cfd1482ea283', commit_message='Upload test_sonuclari.json with huggingface_hub', commit_description='', oid='79db0833106c91d32fed61ef3d59cfd1482ea283', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/chefag/marti-1.5b-veri', endpoint='https://huggingface.co', repo_type='dataset', repo_id='chefag/marti-1.5b-veri'), pr_revision=None, pr_num=None)

## 9. Web Arayüzü: Gradio

> **Konu kartı · Gradio**
> - **Nedir:** Python fonksiyonunu birkaç satırla web arayüzüne çeviren kütüphane; sohbet için hazır `ChatInterface` bileşeni var.
> - **Nasıl çalışır:** `ChatInterface(fn)` → fn(mesaj, geçmiş, …) metin döndürür; Gradio sohbet kutusunu ve sunucuyu kurar. `launch(share=True)` Colab'daki uygulamaya **72 saatlik geçici bir link** verir.
> - **Neden ihtiyaç duyuldu:** Defter hücresi paylaşılamaz; kullanıcı denemeden ürünün nasıl davrandığı anlaşılmaz.
> - **Artıları:** 10 satır; Hugging Face Spaces onu yerleşik destekler; radyo düğmesi gibi bileşenler kolay eklenir.
> - **Eksileri:** Özel tasarım sınırlı; `share` linki Colab kapanınca ölür.

Arayüz kodunu bir yapay zekâ asistanına da yazdırabilirsiniz; mantık (model, RAG, ajan) sizin, vitrin üretken yapay zekânın işi olabilir. Örnek prompt: *"Gradio ChatInterface ile bir sohbet arayüzü yaz. Elimde `sor(mesajlar)`, `asistan_rag(soru)` ((cevap, kaynaklar) döndürür) ve `araclar.ajan(soru, sor)` ((cevap, izler) döndürür) fonksiyonları var. 'Sohbet', 'Rehber (RAG)', 'Ajan' seçenekli bir radyo düğmesi olsun."*

**Bu hücrede:** Üç modlu sohbet. `cevapla` moda göre üç yoldan birini çağırır; RAG'da kaynakları, ajanda araç izlerini cevabın altına ekler. `cache_examples=False` önemli: örnekleri açılışta modelle çalıştırmasın.

In [ ]:
import gradio as gr                               # Web arayüzü (Colab'da kurulu).


def cevapla(mesaj, gecmis, mod):                  # mesaj: son metin; gecmis: önceki turlar; mod: seçim.
    """Moda göre sohbet / RAG / ajan; kaynakları ve araç izlerini cevabın altına ekler."""
    try:                                          # Hata arayüzü çökertmesin:
        if mod == "Sohbet":                       #   düz sohbet (kimlik var, bağlam yok):
            return sor([{"role": "user", "content": mesaj}], 120)   #     tek mesaj.
        if mod == "Rehber (RAG)":                 #   RAG:
            cevap, kaynaklar = asistan_rag(mesaj) #     cevap + kaynaklar,
            return f"{cevap}\n\n_Kaynak: {', '.join(kaynaklar)}_"   #     kaynakları altına yaz.
        cevap, izler = araclar.ajan(mesaj, sor)   #   ajan: cevap + araç izleri,
        return cevap + (f"\n\n_🔧 {' · '.join(izler)}_" if izler else "")   #   izleri altına yaz.
    except Exception as hata:                     # Beklenmedik hata:
        return f"Bir sorun oldu: {type(hata).__name__}. Tekrar dener misiniz? {ISARET}"   # kibar mesaj.


demo = gr.ChatInterface(                          # Hazır sohbet arayüzü:
    cevapla,                                      #   cevaplayan fonksiyon,
    additional_inputs=[gr.Radio(["Sohbet", "Rehber (RAG)", "Ajan"],   #   mod seçimi,
                                value="Rehber (RAG)", label="Mod")],  #   varsayılan RAG,
    title=f"{PROJE_ADI} {ISARET}",                #   başlık,
    description=f"{KURUM} asistanı · {TABAN.split('/')[-1]} + QLoRA",   # açıklama,
    examples=[[r["soru"]] for r in TEST_SORULARI[:3]],   #   örnek sorular (paketten),
    cache_examples=False)                         #   örnekleri açılışta modelle çalıştırma.
demo.launch(share=True, prevent_thread_lock=True)   # Geçici herkese açık link; hücre kilitlenmez.

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eef839d076ffbbb702.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Ne göreceğiz?** Çıktıdaki `https://….gradio.live` linki 72 saat ya da Colab kapanana kadar yaşar. Bu bir **demo**, yayın değil. Kalıcı site için Bölüm 10.

## 10. Yayın: GGUF ve Hugging Face Space

> **Konu kartı · GGUF ve llama.cpp**
> - **Nedir:** llama.cpp, dil modellerini CPU'da verimli çalıştıran bir çalışma zamanı; GGUF onun tek dosyalık model biçimi (ağırlık + tokenizer + sohbet şablonu bir arada).
> - **Nasıl çalışır:** Hugging Face biçimindeki model `convert_hf_to_gguf.py` ile çevrilir ve kuantize edilir (q8_0: 8 bit, kalite kaybı yok denecek kadar az). Python'dan `llama-cpp-python` ile çağrılır.
> - **Neden ihtiyaç duyuldu:** GPU'lu sunucu pahalıdır; GGUF 1.5B modeli 2 çekirdekli CPU'da sohbet için yeterli hızda çalıştırır.
> - **Artıları:** Ucuz donanımda, hatta kendi bilgisayarınızda çalışır; LM Studio ve Ollama da bu biçimi kullanır.
> - **Eksileri:** GPU'ya göre yavaş (3B model 1.5B'den belirgin yavaştır); dönüşüm adımı gerekir.

> **Konu kartı · Hugging Face Spaces**
> - **Nedir:** Hub'daki bir depo türü; içine `app.py` koyarsınız, Hugging Face onu bir sunucuda çalıştırır ve `huggingface.co/spaces/kullanıcı/ad` adresini verir.
> - **Nasıl çalışır:** `README.md` üst bilgisi SDK'yı (gradio) söyler; `requirements.txt` paketleri kurar; uygulama açılır. Temel donanım: 2 vCPU, 16 GB RAM; 48 saat ziyaret olmazsa uyur, ilk ziyarette uyanır.
> - **Neden ihtiyaç duyuldu:** Sunucu kiralamadan, Docker yazmadan model ve uygulamanın aynı yerde durması.
> - **Artıları:** Sunucu yönetimi yok; Git ile sürümlü; GGUF deposundan doğrudan indirir.
> - **Eksileri:** Gradio Space açmak **PRO üyelik** ister (ücretsiz hesapta `402 Payment Required`); CPU'da cevaplar 10–60 saniye sürer.

**Ücretsiz hesaplar için:** Hugging Face, Gradio Space'lerini ücretsiz hesapta açtırmıyor. Model, veri seti ve GGUF depoları ücretsizdir. Bu yüzden son hücre önce Space açmayı dener; hesabınız PRO değilse Space dosyalarınızı GGUF deponuzun `space/` klasörüne yükler. Eğitmen bu dosyalardan sitenizi kendi hesabında yayınlar.

### 10.1 GGUF'a çevirmek

**Bu hücrede:** llama.cpp deposunu indiriyor (yalnızca dönüştürme betiği için), birleştirilmiş modeli q8_0 GGUF'a çeviriyor ve GGUF deposuna yüklüyoruz. 1.5B: ~1.6 GB, 3B: ~3.3 GB.

In [ ]:
from huggingface_hub import snapshot_download     # Depoyu yerel klasöre indirir.

!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
if not os.path.isdir(MODEL_ADI):                  # Model diskte yoksa (yeni oturum):
    snapshot_download(TAM_MODEL, local_dir=MODEL_ADI)   #   Hub'dan indir.
GGUF_DOSYA = f"{MODEL_ADI.lower()}-q8_0.gguf"     # Çıktı dosyasının adı.
# HF biçimi → GGUF; --outtype q8_0 → 8 bit kuantize. Son iki satır özet.
!python llama.cpp/convert_hf_to_gguf.py {MODEL_ADI} --outfile {GGUF_DOSYA} --outtype q8_0 2>&1 | tail -2
api.create_repo(GGUF_DEPO, exist_ok=True, private=False)   # Herkese açık: site indirebilsin.
api.upload_file(path_or_fileobj=GGUF_DOSYA, path_in_repo=GGUF_DOSYA, repo_id=GGUF_DEPO)   # Yükle.
print(f"GGUF: https://huggingface.co/{GGUF_DEPO} | {os.path.getsize(GGUF_DOSYA) / 1e9:.2f} GB")   # Özet.

Writing: 100%|██████████| 1.64G/1.64G [00:08<00:00, 192Mbyte/s]
INFO:hf-to-gguf:Model successfully exported to marti-1.5b-q8_0.gguf


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...tent/marti-1.5b-q8_0.gguf:   4%|3         | 64.6MB / 1.65GB            

Validating                    :           |   0%            

GGUF: https://huggingface.co/chefag/Marti-1.5B-GGUF | 1.65 GB


### 10.2 Site uygulaması: `app.py`

**Bu hücrede:** Sitenin uygulamasını yazıyoruz. Defterdeki mantığın aynısı, iki farkla: (1) model `llama_cpp` ile GGUF'tan yüklenir (CPU), (2) arama Chroma + mpnet yerine **TF-IDF** ile yapılır (1 GB'lık embedding modeli 2 çekirdekli CPU'da ağır; 15 bölümlük bir rehberde TF-IDF yeterli). Kimlik, RAG metni, araçlar ve ajan döngüsü `proje.py` ile `araclar.py`'den gelir: eğitimdekiyle **aynı**.

In [ ]:
%%writefile app.py
"""Bitirme projesi sitesi: GGUF model (CPU) + TF-IDF RAG + araçlı ajan + Gradio."""
import csv                                        # Örnek soruları okumak için.
import re                                         # Rehberi başlıklardan bölmek için.
import gradio as gr                               # Arayüz.
from llama_cpp import Llama                       # GGUF modelini CPU'da çalıştırır.
from pypdf import PdfReader                       # Rehber PDF'i.
from sklearn.feature_extraction.text import TfidfVectorizer   # Harf n-gram sıklığına dayalı arama.
from sklearn.metrics.pairwise import cosine_similarity         # Benzerlik ölçüsü.
import araclar                                    # Araçlar ve ajan döngüsü (defterle aynı dosya).
from proje import PROJE_ADI, KURUM, ISARET, REHBER, TEST, rag_sistem   # Proje kartı.

GGUF_DEPO = "GGUF_DEPO_YER"                       # Yer tutucu: defter deponuzun adını yazar.
GGUF_DOSYA = "GGUF_DOSYA_YER"                     # Yer tutucu: GGUF dosyasının adı.
llm = Llama.from_pretrained(repo_id=GGUF_DEPO, filename=GGUF_DOSYA,   # Hub'dan indir ve yükle:
                            n_ctx=2048, n_threads=2, verbose=False)   # 2048 token bağlam, 2 çekirdek.


def sor(mesajlar, maks=200):                      # Ajan ve RAG bu fonksiyonla modele sorar.
    """Mesaj listesini modele verir, cevabı döndürür (sıcaklık 0: kararlı)."""
    c = llm.create_chat_completion(messages=mesajlar, max_tokens=maks, temperature=0)   # Üret.
    return c["choices"][0]["message"]["content"].strip()   # Metni al.


metin = "\n".join(s.extract_text() for s in PdfReader(REHBER).pages)   # Rehber → metin.
PARCALAR = [b.strip() for b in re.split(r"\n(?=\d+\. [A-ZÇĞİÖŞÜ])", metin)[1:]]   # Bölümler.
tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)).fit(PARCALAR)   # Harf 3-5'lileri.
MATRIS = tfidf.transform(PARCALAR)                # Her bölümün vektörü.


def ara(soru, k=2):                               # soru → en benzer k bölüm.
    """TF-IDF kosinüs benzerliğiyle en yakın k bölümü döndürür."""
    puan = cosine_similarity(tfidf.transform([soru]), MATRIS)[0]   # Soru ile her bölümün benzerliği.
    return [PARCALAR[i] for i in puan.argsort()[::-1][:k]]          # En yüksek k.


araclar.ARA = ara                                 # Ajanın rehberde_ara aracı da TF-IDF kullansın.


def asistan_rag(soru):                            # RAG: ara → bağlam → sor (defterle aynı).
    """(cevap, kaynak başlıkları) döndürür."""
    bulunan = ara(soru)                           # 2 bölüm.
    cevap = sor([{"role": "system", "content": rag_sistem("\n\n".join(bulunan))},   # System,
                 {"role": "user", "content": soru}], 160)                            # soru.
    return cevap, [p.split("\n")[0] for p in bulunan]   # Cevap + başlıklar.


def cevapla(mesaj, gecmis, mod):                  # Gradio'nun çağırdığı fonksiyon.
    """Moda göre sohbet / RAG / ajan."""
    try:                                          # Hata arayüzü çökertmesin.
        if mod == "Sohbet":                       # Düz sohbet:
            return sor([{"role": "user", "content": mesaj}], 120)   #   tek mesaj.
        if mod == "Rehber (RAG)":                 # RAG:
            cevap, kaynaklar = asistan_rag(mesaj) #   cevap + kaynaklar,
            return f"{cevap}\n\n_Kaynak: {', '.join(kaynaklar)}_"   #   kaynakları altına yaz.
        cevap, izler = araclar.ajan(mesaj, sor)   # Ajan: cevap + araç izleri,
        return cevap + (f"\n\n_🔧 {' · '.join(izler)}_" if izler else "")   #   izleri altına yaz.
    except Exception as hata:                     # Beklenmedik hata:
        return f"Bir sorun oldu: {type(hata).__name__}. Tekrar dener misiniz? {ISARET}"   # Kibar mesaj.


with open(TEST, encoding="utf-8") as f:           # Örnek sorular:
    ORNEKLER = [[r["soru"]] for r in list(csv.DictReader(f))[:3]]   #   paketin ilk 3 test sorusu.
demo = gr.ChatInterface(                          # Arayüz (defterdekiyle aynı):
    cevapla,                                      #   cevaplayan fonksiyon,
    additional_inputs=[gr.Radio(["Sohbet", "Rehber (RAG)", "Ajan"],   #   mod seçimi,
                                value="Rehber (RAG)", label="Mod")],  #   varsayılan RAG,
    title=f"{PROJE_ADI} {ISARET}",                #   başlık,
    description=f"{KURUM} asistanı · CPU'da çalışır, cevaplar 10-60 sn sürebilir",   # açıklama,
    examples=ORNEKLER,                            #   örnek sorular,
    cache_examples=False)                         #   açılışta modelle çalıştırma (CPU'da çöker).
if __name__ == "__main__":                        # Space bu dosyayı doğrudan çalıştırır:
    demo.launch()                                 #   sunucuyu başlat.

Writing app.py


### 10.3 Site klasörü

**Bu hücrede:** Siteye gidecek dosyaları `space/` klasöründe topluyoruz: `app.py` (yer tutucular deponuzla değiştirilir), `proje.py`, `araclar.py`, veri dosyaları, `requirements.txt` (llama-cpp-python için hazır CPU paketi adresi; sunucuda derleme beklenmez) ve Space'in `README.md` üst bilgisi.

In [ ]:
import shutil                                     # Dosya kopyalama.
from pathlib import Path                          # Yol işlemleri.

SPACE = Path("space")                             # Siteye yüklenecek klasör.
SPACE.mkdir(exist_ok=True)                        # Yoksa oluştur.
app = Path("app.py").read_text(encoding="utf-8")  # Uygulama metni,
app = app.replace("GGUF_DEPO_YER", GGUF_DEPO).replace("GGUF_DOSYA_YER", GGUF_DOSYA)   # yer tutucular.
(SPACE / "app.py").write_text(app, encoding="utf-8")   # Siteye yaz.
for d in ["proje.py", "araclar.py", REHBER, TABLO, TEST]:   # Ortak kod ve veri dosyaları:
    shutil.copy(d, SPACE / d)                     #   kopyala.
WHEEL = "https://abetlen.github.io/llama-cpp-python/whl/cpu"   # llama-cpp-python hazır CPU paketleri.
(SPACE / "requirements.txt").write_text(          # Paket listesi:
    f"--extra-index-url {WHEEL}\n"                #   derleme yerine hazır paket,
    "llama-cpp-python\nhuggingface_hub\npypdf\nscikit-learn\nrequests\n", encoding="utf-8")
# Space kartı: üst bilgi (SDK, sürüm, giriş dosyası) + kısa açıklama.
(SPACE / "README.md").write_text(f"""---
title: {PROJE_ADI}
emoji: 💬
colorFrom: indigo
colorTo: green
sdk: gradio
sdk_version: {gr.__version__}
app_file: app.py
pinned: false
---
# {PROJE_ADI}
{KURUM} için ince ayarlı asistan. Model: [{GGUF_DEPO}](https://huggingface.co/{GGUF_DEPO}).
Modlar: Sohbet · Rehber (RAG, TF-IDF) · Ajan. CPU'da çalışır; cevaplar 10-60 sn sürebilir.
""", encoding="utf-8")                            # Üst bilgi + açıklama.
print(sorted(p.name for p in SPACE.iterdir()))    # Klasör içeriği.

['README.md', 'app.py', 'araclar.py', 'proje.py', 'rehber.pdf', 'requirements.txt', 'tesisler.csv', 'test_sorulari.csv']


### 10.4 (İsteğe bağlı) Siteyi Colab'da denemek

**Bu hücrede:** Siteye göndermeden önce aynı uygulamayı burada, CPU'da çalıştırıyoruz: `llama-cpp-python` kurulur (~1 dk), GGUF Hub'dan iner, üç moda birer soru sorulur. Sitede de benzer hız beklenir. Zaman kısıtlıysa atlayabilirsiniz.

In [ ]:
import sys                                        # İçe aktarma yolu.
!pip install -q llama-cpp-python --extra-index-url {WHEEL} 2>&1 | tail -1
sys.path.insert(0, str(SPACE))                    # space/ klasörünü içe aktarma yoluna ekle.
uygulama = importlib.import_module("app")         # app.py çalışır: GGUF iner, model yüklenir.
t0 = time.time()                                  # CPU hızını ölçelim.
print(uygulama.cevapla("Sen kimsin?", [], "Sohbet"))                          # Sohbet.
print(uygulama.cevapla(TEST_SORULARI[0]["soru"], [], "Rehber (RAG)"))         # RAG.
print(uygulama.cevapla(ajan_sorulari[0], [], "Ajan"))                         # Ajan.
print(f"3 cevap {time.time() - t0:.0f} sn (CPU)")   # Sitede benzer hız beklenir.
araclar.ARA = chroma_ara                          # Defterdeki ajan yeniden Chroma'yı kullansın.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


./marti-1.5b-q8_0.gguf: reconstructing file:   0%|          |  0.00B / 1.65GB            

./marti-1.5b-q8_0.gguf: downloading bytes:           |  0.00B            

Ben Martı, Martıkent Belediyesi'nin hizmeti asistanı. Benim görevimle ilgili size yardımcı olabilirim. 🕊️
Suyum kesildiğinde, borcun ve 350 TL açma ücretinin ödenmesi gerekir. 🕊️

_Kaynak: 3. Su faturası ve kesinti, 10. Sokak hayvanları ve sahiplendirme_
Çamlık'te tenis türündeki cami koltukları 08:00'dan 21:00'a kadar açık ve aylik ücreti 950 TL'dir. 🕊️

_🔧 tesis_bilgisi(Çamlık Tenis Kortları)_
3 cevap 9 sn (CPU)


### 10.5 Yayın

**Bu hücrede:** Önce Space açmayı deniyoruz. Hesabınız PRO ise site birkaç dakika içinde "Building" → "Running" olur. Değilse Hugging Face `402 Payment Required` döner; o zaman `space/` klasörünüz GGUF deponuza yüklenir ve **eğitmen sitenizi yayınlar**. Her iki durumda da teslim edeceğiniz linkler hücrenin çıktısında yazar.

In [ ]:
SPACE_DEPO = f"{KULLANICI}/{MODEL_ADI.lower()}"   # Space adı (adresin son parçası).
atla = ["__pycache__/*", "*.pyc"]                 # Önbellek dosyaları gitmesin.
try:                                              # Önce kendi Space'inizi deneyin:
    api.create_repo(SPACE_DEPO, repo_type="space",   #   Space deposu aç,
                    space_sdk="gradio", exist_ok=True, private=False)   #   Gradio, herkese açık,
    api.upload_folder(folder_path="space", repo_id=SPACE_DEPO,   #   space/ klasörünü yükle,
                      repo_type="space", ignore_patterns=atla)  #   önbelleksiz.
    SITE = f"https://huggingface.co/spaces/{SPACE_DEPO}"   # Site adresi.
    print("Site:", SITE, "(3-5 dk sonra 'Running'; sorun olursa 'Logs' sekmesi)")   # Bilgi.
except Exception as hata:                         # Hata:
    if "402" not in str(hata):                    #   402 değilse gerçek bir sorun var:
        raise                                     #   hatayı göster.
    api.upload_folder(folder_path="space", path_in_repo="space", repo_id=GGUF_DEPO,   # Space dosyaları
                      ignore_patterns=atla)       #   GGUF deposunun space/ klasörüne.
    SITE = "eğitmen vitrininde yayınlanacak"      # Model kartında yazacak.
    print("Hesabınız Space açamıyor (402: PRO gerekiyor). Site dosyaları:")   # Durum,
    print(f"  https://huggingface.co/{GGUF_DEPO}/tree/main/space")          # dosyaların yeri,
    print("Teslimde GGUF deposu linkini verin; eğitmen sitenizi yayınlayacak.")   # ne yapılacak.

Hesabınız Space açamıyor (402: PRO gerekiyor). Site dosyaları:
  https://huggingface.co/chefag/Marti-1.5B-GGUF/tree/main/space
Teslimde GGUF deposu linkini verin; eğitmen sitenizi yayınlayacak.


**Sorun çıkarsa:** (1) *RUNTIME_ERROR / 502* → `cache_examples=False` duruyor mu? Sayfadaki "Logs" sekmesinde hata satırına bakın. (2) *Build hatası* → Logs'ta hangi paket? `requirements.txt`'yi düzeltip yükleme hücresini yeniden çalıştırın. (3) *Model inmiyor* → GGUF deposu herkese açık mı? (4) *Çok yavaş* → 3B model CPU'da 1.5B'den yaklaşık iki kat yavaştır; cevap uzunluğunu (`maks`) azaltın.

## 11. Model Kartı = Proje Raporu

**Bu hücrede:** Modelinizin Hub sayfasındaki README'yi (model kartı) yazıyoruz. Bu kart **proje raporunuzdur**: ne yaptınız, neden bu taban modeli seçtiniz, veri nasıl üretildi, önce/sonra ölçümleri, test soruları sonuçları, gözlemler ve sınırlar. Sayılar ölçümlerden gelir; **metin kısımlarını siz yazın** (hücrenin başındaki dört değişken).

In [ ]:
from huggingface_hub import hf_hub_download       # Ölçümler yeni oturumda Hub'dan okunur.

# ===== PROJE RAPORU (MODEL KARTI METİNLERİ) =====
PROJE_OZETI = (
    "Martı, Martıkent Belediyesi'nin vatandaş hizmetleri asistanıdır. Vatandaşların Beyaz Masa, "
    "su aboneliği ve kesintileri, emlak vergisi, nikah işlemleri, otoparklar, kütüphaneler, spor tesisleri, "
    "sosyal destek, sokak hayvanları, işyeri ruhsatı, kültür merkezi kiralama, engelli hizmetleri ve atık "
    "toplama konularındaki sorularını yanıtlar. Kurumsal kimliği ve üslubu (🕊️) QLoRA ince ayarıyla modele "
    "kazandırılmış olup; mevzuat ve kurallar RAG (rehber.pdf) üzerinden, tesis saat/ücret bilgileri ile "
    "otopark ücreti hesaplamaları ise Ajan araçları (tesisler.csv, hesapla ve otopark_ucreti) üzerinden "
    "deterministik olarak sağlanır."
)

TABAN_GEREKCESI = (
    "Qwen/Qwen2.5-1.5B-Instruct modeli seçilmiştir. Seçim gerekçeleri: (1) Türkçe dil bilgisi ve yönerge "
    "takibi kapasitesinin 1.5B ölçeğindeki alternatiflere (Llama-3.2-1B vb.) kıyasla daha yüksek olması, "
    "(2) Gemma-2-2B modelinin aksine sohbet şablonunda 'system' rolünü doğrudan desteklemesi (RAG ve Ajan "
    "mimarisinin çalışması için zorunlu), (3) Ücretsiz GPU'da ~1.5 dakikada QLoRA ile eğitilebilmesi ve "
    "GGUF (q8_0) formatında 2 çekirdekli CPU ortamında (Hugging Face Spaces) 3B modellere kıyasla iki kat "
    "daha hızlı yanıt üretebilmesi, (4) Apache-2.0 lisansı ile türev model paylaşımına kısıtlamasız izin vermesidir."
)

GOZLEMLER = (
    "1) Veri ve Eğitim Öncesi/Sonrası: Ham model kimlik sorularında bozuk Türkçe cümleler kurarken ve "
    "'Sen ChatGPT misin?' sorusuna İngilizce politik ret cevabı verirken, 429 sentetik örnekle yapılan "
    "3 epoch QLoRA eğitimi sonrasında test loss değeri 0.622'den 0.282'ye düzenli biçimde düşmüştür. "
    "Eğitim sonrası kimlik adını söyleme oranı 0.00'dan 0.90'a, üslup işareti (🕊️) kullanımı 0.00'dan 1.00'a, "
    "geçerli JSON üretimi 0.79'dan 0.93'e ve doğru araç seçimi 0.21'den 0.86'ya (4 kat artış) yükselmiştir. "
    "2) Bağlamsız vs RAG Kıyası: Bağlamsız sorularda model üslubunu korusa da emlak vergisi için '15 yıl', "
    "otopark için '15 TL' gibi sayılar uydurmuştur. RAG devreye girdiğinde ise '350 TL açma ücreti', "
    "'31 Mayıs' ve 'günlük en fazla 250 TL' gibi rehber bilgilerini doğru vermiş; rehberde olmayan "
    "'Belediye başkanımız kim?' ve 'Metro hattı ne zaman açılacak?' sorularında uydurma yapmayarak "
    "'Bu bilgi rehberde yok' demiştir. "
    "3) Arama (Retrieval) Analizi ve Ajan Performansı: 'Eski koltuğumu nasıl atarım?' ve 'Kafe açacağım, "
    "ruhsat ne kadar sürer?' gibi dolaylı kelimeler içeren sorularda k=2 parça sınırı nedeniyle doğru bölüm "
    "ilk iki sırada gelmediğinde model kurallara sadık kalarak 'rehberde yer almamaktadır' cevabını vermiştir "
    "(model hatası değil, retrieval/arama darlığı). Kendi geliştirdiğimiz 'otopark_ucreti' aracı test sorularında "
    "(3 saat -> 120 TL, 8 saat -> 250 TL tavan ücret, 0.5 saat -> ücretsiz) %100 başarıyla çalışmıştır. "
    "Ayrıca tablo aramalarında modelin Türkçe karakterleri ASCII olarak üretmesi ('Camlik') durumu tespit edilip "
    "araclar.py içerisindeki kucuk() fonksiyonuna Türkçe karakter normalizasyonu eklenerek adım sınırı hatası çözülmüştür."
)

SINIRLAR = (
    "1) Asistan kurum kurallarını ezberlemez; 'Sohbet' modunda (bağlamsız) sorulan mevzuat sorularında "
    "hatalı bilgi uydurabilir, kurallar için mutlaka 'Rehber (RAG)' veya 'Ajan' modu kullanılmalıdır. "
    "2) 1.5B parametreli küçük model, 'Öğrenciyim, Yarı Olimpik Havuz ücreti ne kadar olur?' gibi hem "
    "tablo araması (1200 TL), hem rehberden indirim oranı bulma (%30) hem de hesaplama gerektiren 3 adımlı "
    "kompleks sorularda ilk adımda tesis bilgisini doğru çekip ikinci adımı (indirim hesabı) atlayabilmektedir. "
    "3) Sitedeki TF-IDF araması kelime/harf n-gram benzerliğine dayandığı için eş anlamlı kelimelerde "
    "Chroma (mpnet) kadar esnek değildir. Bu proje kurgusal Martıkent Belediyesi verisiyle eğitim amaçlı üretilmiştir."
)
LISANS = "apache-2.0"

yol = "olcumler.json" if os.path.exists("olcumler.json") else hf_hub_download(
    VERI_SETI, "olcumler.json", repo_type="dataset")
olc = json.load(open(yol, encoding="utf-8"))
ONCE, SONRA = olc["once"], olc["sonra"]
if "SONUCLAR" not in globals():
    yol = hf_hub_download(VERI_SETI, "test_sonuclari.json", repo_type="dataset")
    SONUCLAR = json.load(open(yol, encoding="utf-8"))
    TEST_PUANI = sum(s.get("puan", 0) for s in SONUCLAR) / len(SONUCLAR)
satirlar = "\n".join(f"| {k} | {ONCE[k]:.2f} | {SONRA[k]:.2f} |"
                     for k in ONCE if k != "sure_sn")
test_satirlari = "\n".join(
    f"| {s['soru']} | {s['cevap'][:120].replace('|', '/')} | {s.get('puan', '-')} |"
    for s in SONUCLAR)
site = SITE if "SITE" in globals() else "-"
KART = f"""---
language: [tr]
license: {LISANS}
base_model: {TABAN}
tags: [qlora, turkish, rag, agent, bootcamp]
datasets: [{VERI_SETI}]
---
# {MODEL_ADI} {ISARET}

{PROJE_OZETI}

**Bağlantılar:** [LoRA adaptörü](https://huggingface.co/{ADAPTOR}) ·
[GGUF](https://huggingface.co/{GGUF_DEPO}) ·
[Eğitim verisi](https://huggingface.co/datasets/{VERI_SETI}) · Site: {site}

## Taban model
`{TABAN}`. {TABAN_GEREKCESI}

## Veri ve eğitim
{olc["egitim_ornek"]} sentetik sohbet (kimlik, RAG, ajan; Gemini ile üretildi). QLoRA: r=16, alpha=32,
tüm doğrusal katmanlar, 3 epoch, öğrenme hızı 2e-4.

## Ölçümler (önce → sonra, test seti)
| Ölçüt | Taban | {PROJE_ADI} |
|---|---|---|
{satirlar}

## Test soruları (veri paketi, Gemini hakem: 1 / 0.5 / 0) — ortalama {TEST_PUANI:.2f}
| Soru | Cevap | Puan |
|---|---|---|
{test_satirlari}

## Gözlemler
{GOZLEMLER}

## Sınırlar
{SINIRLAR}
"""
api.upload_file(path_or_fileobj=KART.encode(), path_in_repo="README.md", repo_id=TAM_MODEL)
print(KART[:600], "…")
print("Model kartı:", f"https://huggingface.co/{TAM_MODEL}")

---
language: [tr]
license: apache-2.0
base_model: Qwen/Qwen2.5-1.5B-Instruct
tags: [qlora, turkish, rag, agent, bootcamp]
datasets: [chefag/marti-1.5b-veri]
---
# Marti-1.5B 🕊️

Martı, Martıkent Belediyesi'nin vatandaş hizmetleri asistanıdır. Vatandaşların Beyaz Masa, su aboneliği ve kesintileri, emlak vergisi, nikah işlemleri, otoparklar, kütüphaneler, spor tesisleri, sosyal destek, sokak hayvanları, işyeri ruhsatı, kültür merkezi kiralama, engelli hizmetleri ve atık toplama konularındaki sorularını yanıtlar. Kurumsal kimliği ve üslubu (🕊️) QLoRA ince ayarıyla modele kazandırılmış olup; mevz …
Model kartı: https://huggingface.co/chefag/Marti-1.5B


## Teslim

Teslim formuna şu linkleri girin:

| Ne | Nerede |
|---|---|
| Colab defteriniz | `https://colab.research.google.com/drive/1XDLQwY5nJHOAREf9oeTYsgnGMIu7t19S?usp=sharing` |
| Model (kart = rapor) | `https://huggingface.co/chefag/Marti-1.5B` |
| GGUF deposu | `https://huggingface.co/chefag/Marti-1.5B-GGUF` |
| Eğitim verisi | `https://huggingface.co/datasets/chefag/marti-1.5b-veri` |
| Site | `https://huggingface.co/chefag/Marti-1.5B-GGUF/tree/main/space` |

**Son kontrol:** Defter baştan sona hatasız çalışıyor mu? Model kartında dört paragraf sizin mi? Proje kartındaki ad, kimlik ve dosyalar sizin paketinize ait mi? Kendi aracınız eğitim verisinde ve sitede çalışıyor mu?